# PMHM: spectral–temporal analysis of graded arm occlusion (Phases A–E, steps S0–S14)

**Objective.** This notebook implements the PMHM implementation plan from step S0 to step S14, using a single standalone dataset file, `spectralpaca_pmhm_dataset.parquet` (SPECTRALPACA, Ayala et al., *Sci Data* 2024, CC BY 4.0). It characterises how visible-range spectral features of palm skin respond to graded cuff pressure (20–160 mmHg). It tests hypothesis **H1**: haemoglobin-sensitive ratios fall with pressure. It tests hypothesis **H2**: ratio features generalise across participants better than raw intensities. It quantifies agreement between the camera and the spectrometer reference.

**Data scope.**
- Camera spectral time series exist for participants 01 and 02 (7 tracked ROIs, 25 Hz, 17 protocol blocks).
- Spectrometer reference spectra exist for all 10 participants, at baseline and 160 mmHg only.
- The file also holds calibration records: filter curves, the xenon spectrum, dark statistics and colour-checker values.
- Camera features are **ROI mean/SD statistics of 20 × 20 px regions**. They retain no pixel-level spatial information, so no spatial CNN or image reconstruction is attempted.

**Methodology** (one plan step per pipeline section, run in dependency order):

| Phase | Steps | Content |
|---|---|---|
| A Foundation | S0–S2 | Environment and manifest; contract tests and golden tests reproducing the proposal's numbers; frozen analysis plan |
| B Signal conditioning | S3–S7 | QC and frame flags; dark-offset scenarios; spectrometer→camera forward model; baseline-referenced reflectance and spectrometer-anchored gain; leave-one-participant-out cross-talk correction |
| C Features | S8–S9 | Raw, relative, L1-shape, band-ratio and modified Beer–Lambert features; block windows; smallest detectable change (moving-block bootstrap) |
| D Physiology | S10–S12 | Dose–response and H1 (exact permutation trend + placebo drift control; independent 10-participant spectrometer arm); recovery kinetics (RQ2); feature-family robustness (RQ3) |
| E Agreement and models | S13–S14 | Bland–Altman (incl. repeated-measures), ICC(2,1), spectral angle and repeatability; occlusion-state classification and pressure regression with temporally blocked within-participant CV, cross-participant validation, leave-one-participant-out spectrometer task, H2 test, null/time-only/drift controls, gated TCN/GRU models |

**System architecture and execution flow.**
- Cell 2 holds the whole pipeline. Its **first block is the only place to edit**: dataset and output paths for Colab, Kaggle and local runs, `QUICK_TEST`, `EXPERIMENTS`, `RESUME`, seeds, feature sets, models and training hyper-parameters.
- The pipeline detects the environment, mounts Drive or searches `/kaggle/input`, and validates the file's schema, record types and SHA-256.
- Before execution it writes a timestamped configuration snapshot. It then runs the steps through a checkpointed step engine: every step's outputs are cached and resumed when the configuration hash matches.
- Failures are logged with full tracebacks to `logs/` and a JSONL event stream.

**Validation principles.** Data never leak across time or participants:
- within-participant splits are block-thirds with a 5 s purge;
- cross-participant models train on one participant and test on the other;
- the spectrometer task uses leave-one-participant-out validation;
- hyper-parameters are tuned only inside training folds.

Uncertainty comes from moving-block or participant-level bootstraps.

**Scientific limits.**
- These are healthy volunteers. The results are method-development findings, not evidence of PAD diagnostic validity or clinical StO₂ measurement.
- With two camera participants, camera results are within-participant findings. Population claims rest on the ten-participant spectrometer arm.
- Pressure rises monotonically with time in this protocol. Placebo and drift checks address this confound but cannot remove it.

**Outputs** (under the configured output directory):
- `steps/`: cached step results
- `tables/`: CSV and Parquet results, model results as JSONL
- `predictions/`
- `models/` and `checkpoints/`
- `figures/`: PNG and PDF
- `analysis_plan/`: frozen plan and amendments
- `snapshots/`
- `logs/`
- `reports/`: human-readable Markdown report and machine-readable `experiment_summary_*.json`

In [ ]:
# ===== SECTION: 0. USER CONFIGURATION (all paths and switches; edit only here) =====
DATA_FILENAME = "spectralpaca_pmhm_dataset.parquet"
PATHS = {
    "colab": {
        "data": "/content/drive/MyDrive/PMHM/spectralpaca_pmhm_dataset.parquet",
        "search_root": "/content/drive/MyDrive",
        "output": "/content/drive/MyDrive/PMHM/pmhm_outputs",
    },
    "kaggle": {
        "data": "/kaggle/input/spectralpaca-pmhm/spectralpaca_pmhm_dataset.parquet",
        "search_root": "/kaggle/input",
        "output": "/kaggle/working/pmhm_outputs",
    },
    "local": {
        "data": "./spectralpaca_pmhm_dataset.parquet",
        "search_root": ".",
        "output": "./pmhm_outputs",
    },
}
ENV_OVERRIDE = None                 # None = auto-detect; or "colab" | "kaggle" | "local"
EXPECTED_SHA256 = "eb77b66a76f27eab5a053fe564954d9fe4ef584423f1053cb46542ac38961226"
VERIFY_SHA256 = True
COPY_TO_LOCAL_DISK = True           # Colab: copy the file from Drive to /content for faster reads

QUICK_TEST = False                  # True = smoke-test settings (small bootstraps, short training)
RESUME = True                       # reuse completed steps / experiments with an identical configuration
EXPERIMENTS = "all"                 # "all" or a list of steps, e.g. ["S10", "S14"]; dependencies run automatically
RUN_NAME = None                     # None = "quick" or "full"
SEED = 20261115
DEVICE = "auto"                     # "auto" | "cpu" | "cuda"
AUTO_INSTALL = True                 # pip-install missing light dependencies (statsmodels)
STOP_ON_ERROR = True                # False = record failed non-critical steps and continue
SHOW_FIGURES = False
PLAN_SIGNED_OFF = False             # set True once the supervisor has signed off the frozen analysis plan (S2)
ROI_CORRESPONDENCE_CONFIRMED = False  # True only if spectrometer ROI k is confirmed to be camera ROI k

FEATURE_SETS = ["raw", "rel", "l1", "ratio", "mbll", "all"]
FULL_LADDER_SETS = ["raw", "ratio", "all"]
CLASSICAL_MODELS = ["majority", "logreg_pca", "pls", "rf", "svm"]
H2_MODEL = "logreg_pca"
ENABLE_TEMPORAL = True
TEMPORAL_BACKBONES = ["tcn", "gru"]
TEMPORAL_FEATURE_SET = "ratio"
TRAIN = dict(epochs=40, batch_size=64, lr=1e-3, weight_decay=1e-4, patience=6, hidden=32, dropout=0.1,
             seeds=[0, 1, 2], max_params=100_000)
RF_TREES = 200
BOOT = 2000

# ===== SECTION: 1. IMPORTS AND DEPENDENCY CHECKS =====
import os, sys, json, time, math, hashlib, logging, traceback, platform, subprocess, importlib, importlib.util
import itertools, warnings, shutil, gc, random, datetime as dt
try:
    import resource
except ImportError:
    resource = None
from pathlib import Path

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")


def ensure_pkg(mod, pip_name=None, required=True, install=True):
    try:
        return importlib.import_module(mod)
    except ImportError:
        if install and AUTO_INSTALL:
            print(f"[deps] installing {pip_name or mod}")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or mod], check=False)
            importlib.invalidate_caches()
            try:
                return importlib.import_module(mod)
            except ImportError:
                pass
        if required:
            raise ImportError(f"Required package '{mod}' is missing; install '{pip_name or mod}'.")
        print(f"[deps] optional package '{mod}' unavailable")
        return None


for _m, _p in [("numpy", None), ("pandas", None), ("pyarrow", None), ("scipy", None),
               ("sklearn", "scikit-learn"), ("matplotlib", None)]:
    ensure_pkg(_m, _p)
SM = ensure_pkg("statsmodels", required=False)
PSUTIL = ensure_pkg("psutil", required=False, install=False)
TORCH = ensure_pkg("torch", required=False, install=False) if ENABLE_TEMPORAL else None

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute
import scipy
from scipy import stats, optimize
import matplotlib
IN_NOTEBOOK = "ipykernel" in sys.modules
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (balanced_accuracy_score, f1_score, confusion_matrix, mean_squared_error,
                             mean_absolute_error, r2_score)
from sklearn.model_selection import GroupKFold
if SM is not None:
    import statsmodels.formula.api as smf
if TORCH is not None:
    import torch
    import torch.nn as nn

CODE_VERSION = "pmhm-phase-a-e-1.0"

if QUICK_TEST:
    BOOT, RF_TREES = 200, 40
    FEATURE_SETS, FULL_LADDER_SETS = ["raw", "ratio", "all"], ["ratio"]
    TRAIN.update(epochs=3, patience=2, seeds=[0])

AP = dict(
    gap_s=1.0, plateau_s=30.0, local_ref_s=20.0, baseline_ref_s=60.0, slope_s=10.0, overshoot_s=15.0,
    sdc_win_s=30, sdc_block_s=10, sdc_boot=30 if QUICK_TEST else 200, boot=BOOT,
    kin_boot=10 if QUICK_TEST else 100, mbb_block=30,
    dark_scenarios={"D0": 0.0, "D1": 1.0, "D2": 2.0}, primary_dark="D1",
    disp_flag_px=1.0, sensor_max=1023.0, sat_frac=0.95,
    main_lobe_fwhm=1.5, ridge_alphas=[1e-8, 1e-6, 1e-4, 1e-2, 1.0, 100.0],
    gain_err_tol=0.15, gain_min_bands=12, xt_min_folds=8,
    win_s=5.0, stride_s=3.0 if QUICK_TEST else 1.0, purge_s=5.0,
    tau_bounds=[0.5, 120.0], fit_r2_min=0.5, fit_accept_share=0.7, recovery_t=20.0,
    alpha=0.05, mbll_cond_max=1e3, h1_min_pressure=60, rq2_min_pressure=60,
    primary_feature="r550_609",
    key_features=["r550_609", "rel_550", "rel_609", "mbll_total", "mbll_diff"],
)
GRIDS = {"majority": [{}], "logreg_pca": [{"C": 0.1}, {"C": 1.0}, {"C": 10.0}],
         "pls": [{"n": 2}, {"n": 4}, {"n": 8}], "rf": [{"max_depth": None}, {"max_depth": 8}],
         "svm": [{"C": 0.3}, {"C": 3.0}]}
if QUICK_TEST:
    GRIDS = {k: v[:1] for k, v in GRIDS.items()}

# ===== SECTION: 2. ENVIRONMENT, RESOURCES AND DETERMINISM =====
def detect_env():
    if ENV_OVERRIDE:
        return ENV_OVERRIDE
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "colab"
    except (ModuleNotFoundError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").is_dir():
        return "kaggle"
    return "local"


def mem_mb():
    if PSUTIL is not None:
        vm = PSUTIL.virtual_memory()
        return vm.total / 2**20, vm.available / 2**20
    info = {}
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                k, v = line.split(":")
                info[k] = float(v.split()[0]) / 1024
        return info.get("MemTotal", float("nan")), info.get("MemAvailable", float("nan"))
    except OSError:
        return float("nan"), float("nan")


def rss_mb():
    if PSUTIL is not None:
        return PSUTIL.Process().memory_info().rss / 2**20
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS"):
                    return float(line.split()[1]) / 1024
    except OSError:
        pass
    return float("nan")


def peak_rss_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 if resource else float("nan")


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed % 2**32)
    if TORCH is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)


def select_device():
    if TORCH is None:
        return "none"
    if DEVICE == "cuda" or (DEVICE == "auto" and torch.cuda.is_available()):
        try:
            x = torch.randn(64, 8, 16, device="cuda")
            torch.nn.Conv1d(8, 8, 3).cuda()(x).sum().item()
            return "cuda"
        except Exception as e:
            print(f"[device] CUDA present but unusable ({e!r}); falling back to CPU")
    return "cpu"


def rng(tag):
    h = int(hashlib.sha256(f"{SEED}:{tag}".encode()).hexdigest()[:8], 16)
    return np.random.default_rng(h)


ENV = detect_env()
if ENV == "colab" and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
TOTAL_MB, AVAIL_MB = mem_mb()
DEV = select_device()
PROFILE = "S" if (AVAIL_MB == AVAIL_MB and AVAIL_MB < 1500) else ("L" if DEV == "cuda" else "M")
N_JOBS = max(1, min(4, os.cpu_count() or 1)) if PROFILE == "S" else max(1, (os.cpu_count() or 2) - 1)
if TORCH is not None:
    torch.set_num_threads(min(4, os.cpu_count() or 1) if PROFILE == "S" else N_JOBS)
set_determinism(SEED)

# ===== SECTION: 3. OUTPUT DIRECTORIES AND STRUCTURED LOGGING =====
RUN_MODE = "quick" if QUICK_TEST else "full"
OUT = Path(PATHS[ENV]["output"]).expanduser().resolve() / (RUN_NAME or RUN_MODE)
DIRS = {k: OUT / k for k in ["logs", "snapshots", "steps", "figures", "tables", "models", "checkpoints",
                             "predictions", "reports", "analysis_plan"]}
for _d in DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)
TS = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE = DIRS["logs"] / f"run_{TS}.log"
EVENTS_FILE = DIRS["logs"] / f"events_{TS}.jsonl"
log = logging.getLogger("pmhm")
log.handlers.clear()
log.setLevel(logging.INFO)
_fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
for _h in [logging.StreamHandler(sys.stdout), logging.FileHandler(LOG_FILE)]:
    _h.setFormatter(_fmt)
    log.addHandler(_h)
log.propagate = False
logging.captureWarnings(True)
logging.getLogger("py.warnings").handlers = [logging.FileHandler(LOG_FILE)]


def jdefault(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (Path, dt.datetime)):
        return str(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return str(o)


def clean(o):
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, float) and not math.isfinite(o):
        return None
    if isinstance(o, np.floating):
        return None if not np.isfinite(o) else float(o)
    return o


def wjson(path, obj):
    Path(path).write_text(json.dumps(clean(obj), indent=2, default=jdefault))


def event(step, kind, **kw):
    rec = dict(ts=dt.datetime.now().isoformat(timespec="seconds"), step=step, event=kind, rss_mb=round(rss_mb(), 1), **kw)
    with open(EVENTS_FILE, "a") as f:
        f.write(json.dumps(clean(rec), default=jdefault) + "\n")


log.info(f"PMHM Phase A–E pipeline {CODE_VERSION} | env={ENV} | mode={RUN_MODE} | profile={PROFILE} | "
         f"device={DEV} | RAM total={TOTAL_MB:.0f} MB avail={AVAIL_MB:.0f} MB | out={OUT}")

# ===== SECTION: 4. DATASET FETCHING AND VALIDATION (Colab / Kaggle / local) =====
REQUIRED_COLUMNS = ["record_type", "subject_id", "roi_id", "frame_index", "time_delta_s", "stage", "pressure_mmhg",
                    "phase", "stage_block_index", "has_image_frame", "wavelength_nm", "wavelength_index", "value",
                    "spectrum_index", "channel_index", "dark_mean", "peak_wavelength_nm", "fwhm_nm"]
EXPECTED_COUNTS = {"roi_frame": 125323, "frame": 48157, "spectrometer": 4397250, "camera_filter": 9616,
                   "light_source": 2033, "color_checker": 3840, "channel": 16, "stage": 10, "subject": 10}


def _search(root):
    root = Path(root).expanduser()
    if not root.exists():
        return []
    return sorted(p for p in root.rglob(DATA_FILENAME) if p.is_file())


def _resolve(cfg):
    p = Path(cfg["data"]).expanduser()
    if p.is_file():
        return p
    log.warning(f"Configured data path not found: {p}. Searching under {cfg['search_root']} ...")
    hits = _search(cfg["search_root"])
    if not hits:
        raise FileNotFoundError(f"'{DATA_FILENAME}' not found at {p} or under {cfg['search_root']}. Upload the file "
                                f"or edit PATHS['{ENV}']['data'] in the configuration block.")
    if len(hits) > 1:
        log.warning(f"Several copies found; using {hits[0]}. Others: {[str(h) for h in hits[1:]]}")
    return hits[0]


def fetch_colab(cfg):
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    src = _resolve(cfg)
    if not COPY_TO_LOCAL_DISK:
        return src
    dst = Path("/content/pmhm_data") / DATA_FILENAME
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        log.info(f"Copying {src} -> {dst}")
        shutil.copy2(src, dst)
    return dst


def fetch_kaggle(cfg):
    return _resolve(cfg)


def fetch_local(cfg):
    return _resolve(cfg)


def sha256_file(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def validate_dataset(path):
    info = {"path": str(path), "size_mb": round(path.stat().st_size / 2**20, 2)}
    try:
        pf = pq.ParquetFile(path)
    except Exception as e:
        raise RuntimeError(f"Cannot open {path} as Parquet: {e}") from e
    sch = pf.schema_arrow
    info.update(rows=pf.metadata.num_rows, row_groups=pf.metadata.num_row_groups, columns=len(sch.names))
    missing = [c for c in REQUIRED_COLUMNS if c not in sch.names]
    if missing:
        raise ValueError(f"Dataset is missing required columns: {missing}")
    meta = {k.decode(): v.decode() for k, v in (sch.metadata or {}).items()}
    if "pmhm.camera" not in meta:
        raise ValueError("File-level metadata 'pmhm.camera' missing: not the PMHM standalone dataset?")
    vc = pa.compute.value_counts(pq.read_table(path, columns=["record_type"]).column(0))
    rt = {d["values"]: int(d["counts"]) for d in vc.to_pylist()}
    info["record_counts"] = rt
    unknown = set(rt) - set(EXPECTED_COUNTS)
    if unknown:
        raise ValueError(f"Unexpected record types: {unknown}")
    info["sha256"] = sha256_file(path) if VERIFY_SHA256 else "not-verified"
    if VERIFY_SHA256 and EXPECTED_SHA256 and info["sha256"] != EXPECTED_SHA256:
        raise ValueError(f"SHA-256 mismatch: {info['sha256']} != expected {EXPECTED_SHA256}")
    log.info(f"Dataset OK: {path} | {info['size_mb']} MB | {info['rows']:,} rows | {info['columns']} columns | "
             f"sha256 {info['sha256'][:12]}…")
    log.info("Record types: " + ", ".join(f"{k}={v:,}" for k, v in sorted(info["record_counts"].items())))
    return info, meta


DATA_PATH = {"colab": fetch_colab, "kaggle": fetch_kaggle, "local": fetch_local}[ENV](PATHS[ENV])
DATA_INFO, FILE_META = validate_dataset(DATA_PATH)

# ===== SECTION: 5. STEP ENGINE, CHECKPOINTS AND RESUME =====
STEP_ORDER = [f"S{i}" for i in range(15)]
STEPS, RESULTS = {}, {}
CONFIG_SNAPSHOT = dict(code=CODE_VERSION, seed=SEED, quick=QUICK_TEST, AP=AP, GRIDS=GRIDS, FEATURE_SETS=FEATURE_SETS,
                       FULL_LADDER_SETS=FULL_LADDER_SETS, CLASSICAL_MODELS=CLASSICAL_MODELS, H2_MODEL=H2_MODEL,
                       ENABLE_TEMPORAL=ENABLE_TEMPORAL, TEMPORAL_BACKBONES=TEMPORAL_BACKBONES,
                       TEMPORAL_FEATURE_SET=TEMPORAL_FEATURE_SET, TRAIN=TRAIN, RF_TREES=RF_TREES,
                       ROI_CORRESPONDENCE_CONFIRMED=ROI_CORRESPONDENCE_CONFIRMED, data_sha=DATA_INFO["sha256"])
CONFIG_HASH = hashlib.sha256(json.dumps(clean(CONFIG_SNAPSHOT), sort_keys=True, default=jdefault).encode()).hexdigest()[:16]


def step(name, deps=(), critical=True, title=""):
    def deco(fn):
        try:
            import inspect
            src = hashlib.sha256(inspect.getsource(fn).encode()).hexdigest()[:12]
        except (OSError, TypeError):
            src = "nosource"
        STEPS[name] = dict(fn=fn, deps=list(deps), critical=critical, title=title, src=src)
        return fn
    return deco


def step_key(name):
    parts = [CONFIG_HASH, STEPS[name]["src"]] + [step_key(d) for d in STEPS[name]["deps"]]
    return hashlib.sha256("|".join(parts).encode()).hexdigest()[:16]


def R(tables=None, arrays=None, info=None):
    return {"tables": tables or {}, "arrays": arrays or {}, "info": info or {}}


def flat_table(df):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object and df[c].map(lambda v: isinstance(v, (list, tuple, dict, np.ndarray))).any():
            df[c] = df[c].map(lambda v: json.dumps(clean(v), default=jdefault))
    return df


def save_step(name, res):
    d = DIRS["steps"] / name
    d.mkdir(parents=True, exist_ok=True)
    for k, df in res["tables"].items():
        flat_table(df).to_parquet(d / f"{k}.parquet", index=False)
    for k, a in res["arrays"].items():
        np.save(d / f"{k}.npy", a)
    wjson(d / "info.json", res["info"])
    wjson(d / "done.json", {"key": step_key(name), "finished": dt.datetime.now().isoformat(),
                            "tables": list(res["tables"]), "arrays": list(res["arrays"])})


def load_step(name):
    d = DIRS["steps"] / name
    done = json.loads((d / "done.json").read_text())
    return R({k: pd.read_parquet(d / f"{k}.parquet") for k in done["tables"]},
             {k: np.load(d / f"{k}.npy") for k in done["arrays"]},
             json.loads((d / "info.json").read_text()))


def is_done(name):
    f = DIRS["steps"] / name / "done.json"
    return f.exists() and json.loads(f.read_text()).get("key") == step_key(name)


STATUS = {}


def run_step(name):
    if name in RESULTS:
        return RESULTS[name]
    spec = STEPS[name]
    for d in spec["deps"]:
        run_step(d)
        if STATUS.get(d, {}).get("status") == "failed":
            raise RuntimeError(f"{name} cannot run: dependency {d} failed")
    if RESUME and is_done(name):
        RESULTS[name] = load_step(name)
        STATUS[name] = {"status": "resumed", "title": spec["title"]}
        log.info(f"[{name}] resumed from checkpoint ({spec['title']})")
        event(name, "resumed")
        return RESULTS[name]
    log.info(f"[{name}] START {spec['title']}")
    event(name, "start")
    t0 = time.time()
    try:
        res = spec["fn"]()
        save_step(name, res)
        RESULTS[name] = res
        el = time.time() - t0
        STATUS[name] = {"status": "done", "title": spec["title"], "seconds": round(el, 1),
                        "rss_mb": round(rss_mb(), 1), "peak_rss_mb": round(peak_rss_mb(), 1)}
        log.info(f"[{name}] DONE in {el:.1f} s | RSS {rss_mb():.0f} MB | peak {peak_rss_mb():.0f} MB")
        event(name, "done", seconds=el)
        gc.collect()
        return res
    except Exception as e:
        tb = traceback.format_exc()
        STATUS[name] = {"status": "failed", "title": spec["title"], "error": repr(e)}
        log.error(f"[{name}] FAILED: {e!r}\n{tb}")
        event(name, "failed", error=repr(e), traceback=tb)
        (DIRS["logs"] / f"failure_{name}_{TS}.txt").write_text(tb)
        if spec["critical"] or STOP_ON_ERROR:
            raise
        RESULTS[name] = R(info={"_failed": True, "error": repr(e)})
        return RESULTS[name]


def T(name, key):
    return RESULTS[name]["tables"][key]


def I(name):
    return RESULTS[name]["info"]


def A(name, key):
    return RESULTS[name]["arrays"][key]

# ===== SECTION: 6. DATA ACCESS LAYER =====
CAMERA_META = json.loads(FILE_META["pmhm.camera"])
CH_NM = [int(x) for x in CAMERA_META["channel_order_nm"]]
MEAN_COLS = [f"ch{i:02d}_{nm}nm_mean" for i, nm in enumerate(CH_NM)]
STD_COLS = [f"ch{i:02d}_{nm}nm_std" for i, nm in enumerate(CH_NM)]
I550, I609, I641 = CH_NM.index(550), CH_NM.index(609), CH_NM.index(641)
RAW = [f"raw_{n}" for n in CH_NM]
REL = [f"rel_{n}" for n in CH_NM]
L1 = [f"l1_{n}" for n in CH_NM]
RATIO = [f"r{n}_609" for n in CH_NM if n != 609] + [f"r{n}_641" for n in CH_NM if n != 641]
MBLL = ["mbll_hbo2", "mbll_hb", "mbll_total", "mbll_diff"]
FEATS = RAW + REL + L1 + RATIO + MBLL
FAMILIES = {"raw": RAW, "rel": REL, "l1": L1, "ratio": RATIO, "mbll": MBLL}
SPEC_FAMILIES = {"raw": RAW, "l1": L1, "ratio": RATIO}
ID_COLS = ["subject_id", "roi_id", "frame_index", "t", "block", "phase", "pressure", "prev_pressure", "flag_disp"]
CACHE = {}


def load(record_type, columns=None, filters=None):
    f = [("record_type", "==", record_type)] + list(filters or [])
    return pd.read_parquet(DATA_PATH, columns=columns, filters=f)


def camera_frames():
    if "cam" in CACHE:
        return CACHE["cam"]
    cols = ["subject_id", "roi_id", "frame_index", "time_delta_s", "stage", "pressure_mmhg", "phase",
            "stage_block_index", "preceding_pressure_mmhg", "time_in_block_s", "displacement_px", "roi_patch_max",
            "exposure_ms"] + MEAN_COLS + STD_COLS
    df = load("roi_frame", cols, [("has_image_frame", "==", True)])
    df = df.rename(columns={"time_delta_s": "t", "stage_block_index": "block", "pressure_mmhg": "pressure",
                            "preceding_pressure_mmhg": "prev_pressure"})
    df["roi_id"] = df["roi_id"].astype(int)
    df["block"] = df["block"].astype(int)
    df["pressure"] = df["pressure"].fillna(0).astype(int)
    df["prev_pressure"] = df["prev_pressure"].fillna(0).astype(int)
    for c in MEAN_COLS + STD_COLS + ["displacement_px", "roi_patch_max"]:
        df[c] = df[c].astype(np.float32)
    df = df.sort_values(["subject_id", "roi_id", "t"]).reset_index(drop=True)
    CACHE["cam"] = df
    return df


def channel_table():
    if "ch" not in CACHE:
        CACHE["ch"] = load("channel").dropna(axis=1, how="all").sort_values("channel_index").reset_index(drop=True)
    return CACHE["ch"]


def block_meta(cam):
    g = cam.groupby(["subject_id", "block"])
    bm = g.agg(t0=("t", "min"), t1=("t", "max"), phase=("phase", "first"), pressure=("pressure", "first"),
               prev_pressure=("prev_pressure", "first")).reset_index()
    return bm


def subjects_cam():
    return sorted(camera_frames()["subject_id"].unique())


def load_spectrometer():
    if "spec" in CACHE:
        return CACHE["spec"]
    subs = sorted(load("subject", ["subject_id"])["subject_id"].unique())
    mats, metas, grid = [], [], None
    for s in subs:
        d = load("spectrometer", ["roi_id", "stage", "spectrum_index", "wavelength_index", "wavelength_nm", "value"],
                 [("subject_id", "==", s)])
        d = d.sort_values(["roi_id", "stage", "spectrum_index", "wavelength_index"])
        cnt = d.groupby(["roi_id", "stage", "spectrum_index"]).size()
        if not (cnt == 550).all():
            raise ValueError(f"{s}: spectra with != 550 wavelengths: {cnt[cnt != 550].head()}")
        if grid is None:
            grid = d["wavelength_nm"].to_numpy()[:550].astype(np.float64)
        mats.append(d["value"].to_numpy(np.float32).reshape(-1, 550))
        m = cnt.reset_index()[["roi_id", "stage", "spectrum_index"]]
        m.insert(0, "subject_id", s)
        metas.append(m)
        del d
    meta = pd.concat(metas, ignore_index=True)
    meta["roi_id"] = meta["roi_id"].astype(int)
    meta["is160"] = meta["stage"].str.contains("160").astype(int)
    CACHE["spec"] = (np.vstack(mats), meta, grid)
    return CACHE["spec"]

# ===== SECTION: 7. STATISTICS UTILITIES =====
_PERMS = {}


def perm_spearman(x, y, alternative="two-sided"):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    n = len(x)
    if n < 4:
        return np.nan, np.nan
    rx, ry = stats.rankdata(x), stats.rankdata(y)
    rxc, ryc = rx - rx.mean(), ry - ry.mean()
    den = np.sqrt((rxc ** 2).sum() * (ryc ** 2).sum())
    if den == 0:
        return np.nan, np.nan
    rho = float((rxc * ryc).sum() / den)
    if n <= 8:
        if n not in _PERMS:
            _PERMS[n] = np.array(list(itertools.permutations(range(n))), dtype=np.int8)
        P = ryc[_PERMS[n]]
    else:
        P = np.array([rng(f"perm{n}").permutation(ryc) for _ in range(20000)])
    null = P @ rxc / den
    if alternative == "less":
        p = np.mean(null <= rho + 1e-12)
    elif alternative == "greater":
        p = np.mean(null >= rho - 1e-12)
    else:
        p = np.mean(np.abs(null) >= abs(rho) - 1e-12)
    return rho, float(p)


def holm(p):
    p = np.asarray(p, float)
    out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    idx = np.where(ok)[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]
    m = len(order)
    adj = np.maximum.accumulate([(m - i) * p[j] for i, j in enumerate(order)])
    out[order] = np.minimum(adj, 1.0)
    return out


def boot_ci(x, fn=np.mean, B=None, tag="boot"):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan, np.nan
    r = rng(tag)
    bs = [fn(x[r.integers(0, len(x), len(x))]) for _ in range(B or AP["boot"])]
    return float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def mbb_index(n, block, r):
    if n <= block:
        return r.integers(0, n, n)
    starts = r.integers(0, n - block + 1, int(np.ceil(n / block)))
    return np.concatenate([np.arange(s, s + block) for s in starts])[:n]


def theil(y, x):
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3:
        return np.nan
    return float(stats.theilslopes(y[ok], x[ok])[0])


def icc21(M):
    M = np.asarray(M, float)
    M = M[np.isfinite(M).all(1)]
    n, k = M.shape
    if n < 3:
        return np.nan
    gm = M.mean()
    msr = k * ((M.mean(1) - gm) ** 2).sum() / (n - 1)
    msc = n * ((M.mean(0) - gm) ** 2).sum() / (k - 1)
    sse = ((M - M.mean(1, keepdims=True) - M.mean(0, keepdims=True) + gm) ** 2).sum()
    mse = sse / ((n - 1) * (k - 1))
    return float((msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n))


def bland_altman(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[np.isfinite(d)]
    if len(d) < 2:
        return dict(n=len(d), bias=np.nan, sd=np.nan, loa_lo=np.nan, loa_hi=np.nan)
    m, s = d.mean(), d.std(ddof=1)
    return dict(n=len(d), bias=m, sd=s, loa_lo=m - 1.96 * s, loa_hi=m + 1.96 * s)


def bland_altman_repeated(d, groups):
    df = pd.DataFrame({"d": d, "g": groups}).dropna()
    k = df["g"].nunique()
    n = len(df)
    mi = df.groupby("g").size().to_numpy()
    gm = df.groupby("g")["d"].mean()
    msb = (mi * (gm.to_numpy() - df["d"].mean()) ** 2).sum() / (k - 1)
    msw = ((df["d"] - df["g"].map(gm)) ** 2).sum() / (n - k)
    mh = (n ** 2 - (mi ** 2).sum()) / ((k - 1) * n)
    s2 = max(msb - msw, 0) / mh + msw
    bias = df["d"].mean()
    return dict(n=n, units=k, bias=bias, sd=np.sqrt(s2), loa_lo=bias - 1.96 * np.sqrt(s2),
                loa_hi=bias + 1.96 * np.sqrt(s2))


def spectral_angle(a, b):
    a, b = np.atleast_2d(a), np.atleast_2d(b)
    c = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    return np.degrees(np.arccos(np.clip(c, -1, 1)))


def wilcoxon_exact(x, alternative):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 2 or np.all(x == 0):
        return np.nan
    try:
        return float(stats.wilcoxon(x, alternative=alternative, method="exact").pvalue)
    except Exception:
        return float(stats.wilcoxon(x, alternative=alternative, zero_method="pratt").pvalue)


def md_table(df, floatfmt=".4g", max_rows=60):
    df = df.head(max_rows)
    cols = [str(c) for c in df.columns]
    def f(v):
        if isinstance(v, (float, np.floating)):
            return "" if not np.isfinite(v) else format(v, floatfmt)
        return str(v)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    lines += ["| " + " | ".join(f(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)

# ===== SECTION: 8. OPTICS: FORWARD MODEL, EXTINCTION, CROSS-TALK =====
PRAHL_CSV = """420,480360,407560
422,431880,429880
424,376236,461200
426,326032,481840
428,283112,500840
430,246072,528600
432,214120,552160
434,165332,552160
436,132820,547040
438,119140,501560
440,102580,413280
442,92780,363240
444,81444,282724
446,76324,237224
448,67044,173320
450,62816,103292
452,58864,62640
454,53552,36170
456,49496,30698.8
458,47496,25886.4
460,44480,23388.8
462,41320,20891.2
464,39807.2,19260.8
466,37073.2,18142.4
468,34870.8,17025.6
470,33209.2,16156.4
472,31620,15310
474,30113.6,15048.4
476,28850.8,14792.8
478,27718,14657.2
480,26629.2,14550
482,25701.6,14881.2
484,25180.4,15212.4
486,24669.6,15543.6
488,24174.8,15898
490,23684.4,16684
492,23086.8,17469.6
494,22457.6,18255.6
496,21850.4,19041.2
498,21260,19891.2
500,20932.8,20862
502,20596.4,21832.8
504,20418,22803.6
506,19946,23774.4
508,19996,24745.2
510,20035.2,25773.6
512,20150.4,26936.8
514,20429.2,28100
516,21001.6,29263.2
518,22509.6,30426.4
520,24202.4,31589.6
522,26450.4,32851.2
524,29269.2,34397.6
526,32496.4,35944
528,35990,37490
530,39956.8,39036.4
532,43876,40584
534,46924,42088
536,49752,43592
538,51712,45092
540,53236,46592
542,53292,48148
544,52096,49708
546,49868,51268
548,46660,52496
550,43016,53412
552,39675.2,54080
554,36815.2,54520
556,34476.8,54540
558,33456,54164
560,32613.2,53788
562,32620,52276
564,33915.6,50572
566,36495.2,48828
568,40172,46948
570,44496,45072
572,49172,43340
574,53308,41716
576,55540,40092
578,54728,38467.6
580,50104,37020
582,43304,35676.4
584,34639.6,34332.8
586,26600.4,32851.6
588,19763.2,31075.2
590,14400.8,28324.4
592,10468.4,25470
594,7678.8,22574.8
596,5683.6,19800
598,4504.4,17058.4
600,3200,14677.2
602,2664,13622.4
604,2128,12567.6
606,1789.2,11513.2
608,1647.6,10477.6
610,1506,9443.6
612,1364.4,8591.2
614,1222.8,7762
616,1110,7344.8
618,1026,6927.2
620,942,6509.6
622,858,6193.2
624,774,5906.8
626,707.6,5620
628,658.8,5366.8
630,610,5148.8
632,561.2,4930.8
634,512.4,4730.8
636,478.8,4602.4
638,460.4,4473.6
640,442,4345.2
642,423.6,4216.8
644,405.2,4088.4
646,390.4,3965.08
648,379.2,3857.6
650,368,3750.12
652,356.8,3642.64
654,345.6,3535.16
656,335.2,3427.68
658,325.6,3320.2
660,319.6,3226.56
662,314,3140.28
664,308.4,3053.96
666,302.8,2967.68
668,298,2881.4
670,294,2795.12
672,290,2708.84
674,285.6,2627.64
676,282,2554.4
678,279.2,2481.16
680,277.6,2407.92
682,276,2334.68
684,274.4,2261.48
686,272.8,2188.24
688,274.4,2115
690,276,2051.96
692,277.6,2000.48
694,279.2,1949.04
696,282,1897.56
698,286,1846.08
700,290,1794.28"""
PRAHL_SOURCE = "Prahl S. Tabulated molar extinction coefficient for hemoglobin in water. OMLC, https://omlc.org/spectra/hemoglobin/ (420-700 nm rows, cm-1/M)"


def prahl_table():
    rows = [list(map(float, r.split(","))) for r in PRAHL_CSV.strip().splitlines()]
    return np.array(rows)


def quad_weights(grid):
    g = np.asarray(grid, float)
    q = np.zeros_like(g)
    q[1:-1] = (g[2:] - g[:-2]) / 2
    q[0], q[-1] = (g[1] - g[0]) / 2, (g[-1] - g[-2]) / 2
    return q


def optics_on_grid(grid):
    f = load("camera_filter", ["channel_index", "wavelength_nm", "value"])
    n_dup = int(f.duplicated(["channel_index", "wavelength_nm"]).sum())
    f = f.groupby(["channel_index", "wavelength_nm"], as_index=False)["value"].mean()
    L = load("light_source", ["wavelength_nm", "value"]).groupby("wavelength_nm", as_index=False)["value"].mean()
    L = L.sort_values("wavelength_nm")
    Fm = np.zeros((16, len(grid)))
    trunc = np.zeros(16)
    for c, g in f.groupby("channel_index"):
        g = g.sort_values("wavelength_nm")
        Fm[int(c)] = np.interp(grid, g["wavelength_nm"], g["value"], left=0, right=0)
        wl = g["wavelength_nm"].to_numpy()
        fe = g["value"].to_numpy() * np.interp(wl, L["wavelength_nm"], L["value"])
        inside = (wl >= grid.min()) & (wl <= grid.max())
        trunc[int(c)] = fe[~inside].sum() / fe.sum()
    E = np.interp(grid, L["wavelength_nm"], L["value"])
    return Fm, E, trunc, n_dup


def band_weights(Fm, E, grid, mask=None):
    W = Fm * E[None, :]
    if mask is not None:
        W = W * mask
    Wq = W * quad_weights(grid)[None, :]
    return Wq / Wq.sum(1, keepdims=True)


def main_lobe_mask(grid):
    ch = channel_table()
    pk = ch["peak_wavelength_nm"].to_numpy(float)
    fw = ch["fwhm_nm"].to_numpy(float)
    return (np.abs(grid[None, :] - pk[:, None]) <= AP["main_lobe_fwhm"] * fw[:, None]).astype(float)


def extinction_design(grid, Wn):
    pt = prahl_table()
    eo = np.interp(grid, pt[:, 0], pt[:, 1]) / 1000.0
    ed = np.interp(grid, pt[:, 0], pt[:, 2]) / 1000.0
    Eb = np.c_[Wn @ eo, Wn @ ed, np.ones(16)]
    return Eb, float(np.linalg.cond(Eb))

# ===== SECTION: 9. FEATURE ENGINEERING =====
def compute_features(I, D, Iref, Eb=None):
    I = np.asarray(I, np.float64)
    X = I - D
    Xr = np.asarray(Iref, np.float64) - D
    with np.errstate(divide="ignore", invalid="ignore"):
        rel = X / Xr
        l1 = X / X.sum(1, keepdims=True)
        r609 = np.delete(X / X[:, [I609]], I609, 1)
        r641 = np.delete(X / X[:, [I641]], I641, 1)
        if Eb is not None:
            Aten = -np.log(np.where(rel > 0, rel, np.nan))
            coef = Aten @ np.linalg.pinv(Eb).T
            mb = np.c_[coef[:, 0], coef[:, 1], coef[:, 0] + coef[:, 1], coef[:, 0] - coef[:, 1]]
        else:
            mb = np.full((len(X), 4), np.nan)
    out = np.concatenate([I, rel, l1, r609, r641, mb], 1)
    out[~np.isfinite(out)] = np.nan
    return out.astype(np.float32)


def baseline_ref_counts(cam):
    bm = block_meta(cam)
    refs = {}
    for (s, r), g in cam[cam["block"] == 0].groupby(["subject_id", "roi_id"]):
        t1 = bm[(bm.subject_id == s) & (bm.block == 0)]["t1"].iloc[0]
        refs[(s, r)] = g.loc[g["t"] >= t1 - AP["baseline_ref_s"], MEAN_COLS].to_numpy(np.float64).mean(0)
    return refs


def roi_mean_rows(df, cols):
    n_roi = df.groupby("subject_id")["roi_id"].nunique()
    g = df.groupby(["subject_id", "frame_index"], sort=False)
    m = g[cols].mean()
    cnt = g.size()
    ids = g[["t", "block", "phase", "pressure", "prev_pressure"]].first()
    flag = g["flag_disp"].max()
    out = pd.concat([ids, flag, m], axis=1).reset_index()
    out = out[cnt.to_numpy() == out["subject_id"].map(n_roi).to_numpy()]
    out.insert(1, "roi_id", 0)
    return out[ID_COLS + cols]


def camera_feature_table(cam, D, Eb, M=None, gains=None, rows=None):
    sub = cam if rows is None else cam.loc[rows]
    refs = baseline_ref_counts(cam)
    I = sub[MEAN_COLS].to_numpy(np.float64)
    Iref = np.vstack([refs[k] for k in zip(sub["subject_id"], sub["roi_id"])])
    if M is not None:
        G = np.vstack([gains[s] for s in sub["subject_id"]])
        I = ((I - D) / G) @ M.T * G + D
        Iref = ((Iref - D) / G) @ M.T * G + D
    F = compute_features(I, D, Iref, Eb)
    out = pd.DataFrame(F, columns=FEATS, index=sub.index)
    ids = sub[["subject_id", "roi_id", "frame_index", "t", "block", "phase", "pressure", "prev_pressure"]].copy()
    ids["flag_disp"] = (sub["displacement_px"].fillna(0) > AP["disp_flag_px"]).astype(np.int8)
    return pd.concat([ids, out], axis=1)

# ===== SECTION: 10. BLOCK WINDOWS AND SMALLEST DETECTABLE CHANGE =====
def wmean(t, X, lo, hi):
    m = (t >= lo) & (t <= hi)
    if not m.any():
        return np.full(X.shape[1], np.nan), 0, False
    tt = t[m]
    ok = bool(len(tt) < 2 or np.diff(tt).max() <= AP["gap_s"])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(X[m], 0), int(m.sum()), ok


def block_stats(FF, bm, cols):
    rows = []
    for (s, r), g in FF.groupby(["subject_id", "roi_id"]):
        t = g["t"].to_numpy()
        X = g[cols].to_numpy(np.float64)
        b = bm[bm.subject_id == s].set_index("block")
        b0 = b.loc[0]
        rows.append((s, r, 0, "baseline", 0, 0, "ref_global") + wmean(t, X, b0.t1 - AP["baseline_ref_s"], b0.t1))
        for k, bk in b.iterrows():
            if k == 0:
                continue
            rows.append((s, r, k, bk.phase, bk.pressure, bk.prev_pressure, "block_mean") + wmean(t, X, bk.t0, bk.t1))
            if bk.phase == "clamping":
                rows.append((s, r, k, bk.phase, bk.pressure, bk.prev_pressure, "plateau")
                            + wmean(t, X, bk.t1 - AP["plateau_s"], bk.t1))
                rows.append((s, r, k, bk.phase, bk.pressure, bk.prev_pressure, "plateau_last30")
                            + wmean(t, X, bk.t1 - 30.0, bk.t1))
            else:
                rows.append((s, r, k, bk.phase, bk.pressure, bk.prev_pressure, "rest_ref")
                            + wmean(t, X, bk.t1 - AP["local_ref_s"], bk.t1))
    meta = pd.DataFrame([r[:7] + (r[8], r[9]) for r in rows],
                        columns=["subject_id", "roi_id", "block", "phase", "pressure", "prev_pressure", "window", "n", "valid"])
    vals = pd.DataFrame(np.vstack([r[7] for r in rows]), columns=cols)
    return pd.concat([meta, vals], axis=1)


def deltas(bs, cols):
    out = []
    for (s, r), g in bs.groupby(["subject_id", "roi_id"]):
        gi = g.set_index(["block", "window"])
        rg = gi.loc[(0, "ref_global"), cols].to_numpy(float)
        vg = bool(gi.loc[(0, "ref_global"), "valid"])
        for k in sorted(g.loc[g.phase == "clamping", "block"].unique()):
            pl = gi.loc[(k, "plateau"), cols].to_numpy(float)
            pr = int(gi.loc[(k, "plateau"), "pressure"])
            if k - 1 >= 2:
                rl, vl = gi.loc[(k - 1, "rest_ref"), cols].to_numpy(float), bool(gi.loc[(k - 1, "rest_ref"), "valid"])
            else:
                rl, vl = rg, vg
            v = bool(gi.loc[(k, "plateau"), "valid"])
            out.append((s, r, k, pr, "local", v and vl, pl - rl))
            out.append((s, r, k, pr, "global", v and vg, pl - rg))
            with np.errstate(divide="ignore", invalid="ignore"):
                out.append((s, r, k, pr, "local_pct", v and vl, 100 * (pl - rl) / np.abs(rl)))
                out.append((s, r, k, pr, "global_pct", v and vg, 100 * (pl - rg) / np.abs(rg)))
            if (k + 1, "rest_ref") in gi.index:
                rr = gi.loc[(k + 1, "rest_ref"), cols].to_numpy(float)
                out.append((s, r, k, pr, "placebo", vl and bool(gi.loc[(k + 1, "rest_ref"), "valid"]), rr - rl))
    meta = pd.DataFrame([o[:6] for o in out], columns=["subject_id", "roi_id", "block", "pressure", "contrast", "valid"])
    return pd.concat([meta, pd.DataFrame(np.vstack([o[6] for o in out]), columns=cols)], axis=1)


def bins_1s(t, X, t0):
    b = np.floor(t - t0).astype(int)
    df = pd.DataFrame(X)
    df["b"] = b
    m = df.groupby("b").mean()
    return m.index.to_numpy(), m.to_numpy(np.float64)


def sdc_from_bins(Bm, w):
    Bm = np.where(np.isfinite(Bm), Bm, np.nanmean(Bm, 0, keepdims=True))
    C = np.vstack([np.zeros(Bm.shape[1]), np.cumsum(Bm, 0)])
    pos = Bm.shape[0] - w + 1
    if pos < w + 1:
        return np.full(Bm.shape[1], np.nan)
    means = (C[w:w + pos] - C[:pos]) / w
    i, j = np.triu_indices(pos, k=w)
    return np.percentile(np.abs(means[j] - means[i]), 95, axis=0)


def kinetics(FF, bm, bs, feats):
    rows = []
    for (s, r), g in FF.groupby(["subject_id", "roi_id"]):
        b = bm[bm.subject_id == s].set_index("block")
        gi = bs[(bs.subject_id == s) & (bs.roi_id == r)].set_index(["block", "window"])
        for k, bk in b.iterrows():
            if k == 0:
                continue
            gg = g[(g["block"] == k)]
            tt = gg["t"].to_numpy() - bk.t0
            for f in feats:
                y = gg[f].to_numpy(float)
                rec = dict(subject_id=s, roi_id=r, block=k, phase=bk.phase, pressure=int(bk.pressure),
                           prev_pressure=int(bk.prev_pressure), feature=f)
                m = tt <= AP["slope_s"]
                step_ = max(1, int(m.sum() // 250))
                rec["slope10"] = theil(y[m][::step_], tt[m][::step_])
                bx, by = bins_1s(tt, y[:, None], 0.0)
                by = by[:, 0]
                if bk.phase == "clamping":
                    ref = gi.loc[(k - 1, "rest_ref"), f] if k - 1 >= 2 else gi.loc[(0, "ref_global"), f]
                    fin = gi.loc[(k, "plateau"), f] - ref
                    c = by - ref
                    hit = np.where(c / fin >= 0.9)[0] if fin != 0 and np.isfinite(fin) else []
                    rec["t90"] = float(bx[hit[0]]) if len(hit) else np.nan
                    rec["auc"] = float(np.trapezoid(c, bx)) if hasattr(np, "trapezoid") else float(np.trapz(c, bx))
                    rec["overshoot"] = np.nan
                else:
                    kc = k - 1
                    ref = gi.loc[(kc - 1, "rest_ref"), f] if kc - 1 >= 2 else gi.loc[(0, "ref_global"), f]
                    pl = gi.loc[(kc, "plateau"), f]
                    sgn = np.sign(pl - ref)
                    m15 = bx <= AP["overshoot_s"]
                    rec["overshoot"] = float(max(0.0, np.nanmax(-sgn * (by[m15] - ref)))) if m15.any() else np.nan
                    rec["t90"], rec["auc"] = np.nan, np.nan
                rows.append(rec)
    return pd.DataFrame(rows)

# ===== SECTION: 11. PLOTTING HELPERS =====
C_SUBJ = ["#2a78d6", "#eb6834", "#2f9e44", "#7048e8"]
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 220, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False})
FIGURES = []


def savefig(fig, name, caption):
    p = DIRS["figures"] / f"{name}.png"
    fig.savefig(p, bbox_inches="tight")
    fig.savefig(DIRS["figures"] / f"{name}.pdf", bbox_inches="tight")
    FIGURES.append({"file": str(p), "caption": caption})
    if SHOW_FIGURES and IN_NOTEBOOK:
        plt.show()
    plt.close(fig)
    return str(p)


def shade_clamps(ax, bm, s):
    for _, b in bm[(bm.subject_id == s) & (bm.phase == "clamping")].iterrows():
        ax.axvspan(b.t0, b.t1, color="0.85", lw=0, zorder=0)
        ax.text((b.t0 + b.t1) / 2, 1.0, f"{b.pressure}", transform=ax.get_xaxis_transform(), ha="center",
                va="bottom", fontsize=7, color="0.35")

# ===== SECTION: 12. S0 ENVIRONMENT AND RUN MANIFEST =====
@step("S0", title="Environment and run manifest")
def s0():
    vers = {m: getattr(sys.modules[m], "__version__", "?") for m in
            ["numpy", "pandas", "pyarrow", "scipy", "sklearn", "matplotlib"] if m in sys.modules}
    vers["statsmodels"] = getattr(SM, "__version__", None) if SM else None
    vers["torch"] = getattr(TORCH, "__version__", None) if TORCH else None
    pt = prahl_table()
    info = dict(code=CODE_VERSION, env=ENV, profile=PROFILE, device=DEV, n_jobs=N_JOBS, python=sys.version.split()[0],
                platform=platform.platform(), cpu_count=os.cpu_count(), ram_total_mb=TOTAL_MB, ram_avail_mb=AVAIL_MB,
                gpu=(torch.cuda.get_device_name(0) if DEV == "cuda" else None), versions=vers, config_hash=CONFIG_HASH,
                data=DATA_INFO, prahl_rows=len(pt), prahl_range_nm=[pt[0, 0], pt[-1, 0]],
                prahl_sha256=hashlib.sha256(PRAHL_CSV.encode()).hexdigest(), prahl_source=PRAHL_SOURCE,
                statsmodels_available=SM is not None, torch_available=TORCH is not None)
    if SM is None:
        log.warning("statsmodels unavailable: mixed-model analyses will be reported as not run")
    if TORCH is None and ENABLE_TEMPORAL:
        log.warning("PyTorch unavailable: temporal models will be reported as not run")
    return R(info=info)

# ===== SECTION: 13. S1 DATA ACCESS AND CONTRACT TESTS =====
@step("S1", deps=["S0"], title="Data access layer, contract and golden tests")
def s1():
    tests, cost = [], []
    def chk(name, cond, detail=""):
        tests.append(dict(test=name, passed=bool(cond), detail=str(detail)))
    sch = pq.ParquetFile(DATA_PATH).schema_arrow
    chk("81 columns", len(sch.names) == 81, len(sch.names))
    chk("record counts", DATA_INFO["record_counts"] == EXPECTED_COUNTS, DATA_INFO["record_counts"])
    chk("sha256", (not VERIFY_SHA256) or DATA_INFO["sha256"] == EXPECTED_SHA256, DATA_INFO["sha256"][:16])
    chk("channel columns present", all(c in sch.names for c in MEAN_COLS + STD_COLS))
    for rt in ["roi_frame", "spectrometer", "channel", "subject"]:
        t0 = time.time()
        n = pq.read_table(DATA_PATH, columns=["record_type"], filters=[("record_type", "==", rt)]).num_rows
        cost.append(dict(record_type=rt, rows=n, seconds=round(time.time() - t0, 3), rss_after_mb=round(rss_mb(), 1)))
    allr = load("roi_frame", ["subject_id", "roi_id", "has_image_frame"] + MEAN_COLS[:1])
    cnt = allr.groupby("subject_id").size().to_dict()
    chk("roi_frame per subject", cnt == {"subject_01": 71344, "subject_02": 53979}, cnt)
    noimg = allr[~allr["has_image_frame"]]
    chk("no spectra without image", noimg[MEAN_COLS[0]].isna().all(), f"{len(noimg)} post-video rows")
    del allr
    cam = camera_frames()
    chk("17 blocks per camera subject", (cam.groupby("subject_id")["block"].nunique() == 17).all())
    mono = cam.groupby(["subject_id", "roi_id"])["t"].apply(lambda s: bool((np.diff(s.to_numpy()) >= 0).all()))
    chk("time monotonic per ROI", mono.all())
    ch = channel_table()
    nominal = [int(c.split("_")[1][:-2]) for c in MEAN_COLS]
    chk("channel order matches metadata", nominal == CH_NM and len(ch) == 16)
    cc = load("color_checker", ["source_row_index", "value"]).sort_values("source_row_index")
    sums = cc.groupby(cc["source_row_index"] // 16)["value"].sum()
    chk("colour-checker replicates L1-normalised", np.allclose(sums, 1.0, atol=1e-9), f"{len(sums)} replicates")
    S, meta, grid = load_spectrometer()
    chk("spectra x 550 wavelengths", S.shape == (7995, 550), S.shape)
    # golden tests (proposal v1.0 numbers)
    gold = []
    bm = block_meta(cam)
    for sid, g in cam.groupby("subject_id"):
        r = (g[MEAN_COLS[I550]] / g[MEAN_COLS[I609]]).groupby(g["block"]).mean()
        d = (r / r.loc[0] - 1) * 100
        b = bm[bm.subject_id == sid].set_index("block")
        cl = d[(b.phase == "clamping") & (b.pressure >= 60)]
        m = g.groupby("frame_index").agg(blk=("block", "first"), a=(MEAN_COLS[I550], "mean"))
        a = m.groupby("blk")["a"].mean()
        raw160 = (a[b.index[b.pressure == 160][0]] / a.loc[0] - 1) * 100
        rec = sum(1 for k in range(1, 16, 2) if d.loc[k + 1] > d.loc[k])
        gold.append(dict(subject_id=sid, n_neg=int((cl < 0).sum()), n_stages=len(cl), min_pct=round(cl.min(), 1),
                         max_pct=round(cl.max(), 1), raw550_160_pct=round(raw160, 1), recovered=rec))
    gold = pd.DataFrame(gold).set_index("subject_id")
    exp = {"subject_01": (-4.1, -0.4, -5.7, 5), "subject_02": (-6.1, -2.5, 7.8, 6)}
    for sid, (lo, hi, raw, rec) in exp.items():
        g = gold.loc[sid]
        chk(f"golden 550/609 {sid}", g.n_neg == 6 == g.n_stages and abs(g.min_pct - lo) < 0.051
            and abs(g.max_pct - hi) < 0.051, f"{g.min_pct:.1f}..{g.max_pct:.1f}")
        chk(f"golden raw550@160 {sid}", abs(g.raw550_160_pct - raw) < 0.051, f"{g.raw550_160_pct:.1f}")
        chk(f"golden recovery {sid}", g.recovered == rec, g.recovered)
    band = (grid >= 500) & (grid <= 650)
    subj_mean = pd.DataFrame({"s": meta.subject_id, "is160": meta.is160, "v": S[:, band].mean(1)})
    sm = subj_mean.groupby(["s", "is160"])["v"].mean().unstack()
    n_higher = int((sm[1] > sm[0]).sum())
    gmean = [S[(meta.is160 == k).to_numpy()][:, band].mean(0) for k in (0, 1)]
    pm = pd.DataFrame(S[:, band]).groupby([meta.subject_id.to_numpy(), meta.is160.to_numpy()]).mean()
    grp = [pm.xs(k, level=1).mean(0).to_numpy() for k in (0, 1)]
    rise = (grp[1] / grp[0] - 1) * 100
    chk("golden spectrometer 6/10 higher at 160", n_higher == 6, n_higher)
    med = float(np.median(rise))
    chk("golden spectrometer group rise 500-650 nm (verified: 16.3-30.1%, median 20.8%)",
        abs(rise.min() - 16.3) < 0.15 and abs(rise.max() - 30.1) < 0.15 and abs(med - 20.8) < 0.15,
        f"{rise.min():.1f}..{rise.max():.1f}, median {med:.1f}; proposal v1.0 text '18-22%' understates the range")
    tdf = pd.DataFrame(tests)
    failed = tdf[~tdf.passed]
    log.info(f"Contract/golden tests: {tdf.passed.sum()}/{len(tdf)} passed")
    if len(failed):
        raise AssertionError("Contract or golden tests failed:\n" + failed.to_string())
    return R({"tests": tdf, "golden": gold.reset_index(), "access_cost": pd.DataFrame(cost)},
             info=dict(n_tests=len(tdf), n_passed=int(tdf.passed.sum()), spectro_rise_pct=[rise.min(), rise.max()],
                       spectro_rise_median_pct=med,
                       proposal_erratum="Proposal v1.0 section 5.4 states an 18-22% rise across 500-650 nm; the file "
                                        "gives 16.3-30.1% (median 20.8%)."))

# ===== SECTION: 14. S2 ANALYSIS PLAN FREEZE =====
@step("S2", deps=["S1"], title="Analysis plan frozen before confirmatory steps")
def s2():
    plan = dict(code=CODE_VERSION, analysis_parameters=AP, grids=GRIDS, feature_families=FAMILIES,
                feature_sets=FEATURE_SETS, models=CLASSICAL_MODELS, h2_model=H2_MODEL,
                temporal=dict(enabled=ENABLE_TEMPORAL, backbones=TEMPORAL_BACKBONES, features=TEMPORAL_FEATURE_SET,
                              train=TRAIN),
                decisions=dict(
                    H1_camera="per participant: exact one-sided permutation Spearman p<alpha on ROI-mean local delta of "
                              "r550_609 across 8 pressures; delta < -SDC95 at every pressure >= 60 mmHg; clamp Theil-Sen "
                              "slope more negative and larger in magnitude than placebo slope",
                    H1_spectrometer="exact one-sided Wilcoxon (n=10) on participant mean log(S550/S609) 160 minus "
                                    "baseline, p<alpha and bootstrap CI below 0",
                    H1_overall="supported if both arms; partially if one; not supported if neither",
                    RQ2="Holm-adjusted permutation trend of tau, half-time, overshoot vs preceding pressure (>=60 mmHg) "
                        "p<alpha in both participants",
                    RQ3="family robust if not worse than raw on any criterion and better on >=2",
                    H2="ratio error < raw error in both cross-participant directions (T1a, H2 model) and in T1d "
                       "(exact one-sided Wilcoxon over 10 folds p<alpha)"),
                status=dict(H1_camera_arm="non-confirmatory: generated the hypothesis",
                            H1_spectrometer_arm="independent: ratio never computed before this plan",
                            roi_correspondence_confirmed=ROI_CORRESPONDENCE_CONFIRMED),
                spatial_scope="ROI mean/SD statistics only; no pixel-level spatial information is available")
    h = hashlib.sha256(json.dumps(clean(plan), sort_keys=True, default=jdefault).encode()).hexdigest()[:16]
    frozen = DIRS["analysis_plan"] / "frozen.json"
    amend = None
    if frozen.exists():
        prev = json.loads(frozen.read_text())
        if prev["hash"] != h:
            amend = dict(ts=TS, previous=prev["hash"], new=h)
            with open(DIRS["analysis_plan"] / "amendments.jsonl", "a") as f:
                f.write(json.dumps(amend) + "\n")
            log.warning(f"Analysis plan changed after freeze ({prev['hash']} -> {h}); amendment logged")
    wjson(DIRS["analysis_plan"] / f"analysis_plan_{h}.json", plan)
    if not frozen.exists() or amend:
        wjson(frozen, dict(hash=h, frozen_at=TS))
    if not PLAN_SIGNED_OFF:
        log.warning("PLAN_SIGNED_OFF=False: confirmatory results (S10-S14) will be labelled 'plan not signed off'")
    return R(info=dict(plan_hash=h, signed_off=PLAN_SIGNED_OFF, amendment=amend, plan=plan))

# ===== SECTION: 15. S3 QUALITY CONTROL AND FRAME FLAGS =====
@step("S3", deps=["S2"], title="Quality control and frame flags")
def s3():
    cam = camera_frames()
    allr = load("roi_frame", ["subject_id", "roi_id", "has_image_frame"])
    post = allr.groupby("subject_id")["has_image_frame"].apply(lambda s: int((~s).sum())).rename("post_video_rows")
    fr = load("frame", ["subject_id", "frame_index", "time_delta_s", "stage", "exposure_ms"])
    gaps, qc = [], []
    for s, g in fr.sort_values(["subject_id", "frame_index"]).groupby("subject_id"):
        t = g["time_delta_s"].to_numpy()
        d = np.diff(t)
        for i in np.where(d > AP["gap_s"])[0]:
            gaps.append(dict(subject_id=s, gap_start_s=t[i], gap_end_s=t[i + 1], gap_s=d[i]))
        qc.append(dict(subject_id=s, n_frames=len(g), fps=1 / np.median(d), interval_cv=np.std(d[d <= AP["gap_s"]]) /
                       np.mean(d[d <= AP["gap_s"]]), exposure_ms=g["exposure_ms"].median(),
                       has_roi_spectra=s in cam["subject_id"].unique()))
    m = cam[["subject_id", "frame_index", "stage"]].merge(fr[["subject_id", "frame_index", "stage"]],
                                                          on=["subject_id", "frame_index"], suffixes=("", "_fr"))
    stage_agree = float((m["stage"] == m["stage_fr"]).mean())
    ch = channel_table()
    noise = []
    for (s, r), g in cam.groupby(["subject_id", "roi_id"]):
        X = g[MEAN_COLS].to_numpy(np.float64)
        dX = np.diff(X, axis=0) / np.sqrt(2)
        same = np.diff(g["block"].to_numpy()) == 0
        tn = np.nanstd(dX[same], 0)
        for i, nm in enumerate(CH_NM):
            noise.append(dict(subject_id=s, roi_id=r, nm=nm, mean_counts=X[:, i].mean(),
                              within_roi_sd=float(g[STD_COLS[i]].mean()), temporal_noise=tn[i],
                              dark_sd=float(ch.loc[i, "dark_std"]), snr=X[:, i].mean() / tn[i]))
    flags = cam.groupby(["subject_id", "roi_id"]).agg(
        n=("t", "size"), disp_flag=("displacement_px", lambda x: int((x > AP["disp_flag_px"]).sum())),
        disp_mean=("displacement_px", "mean"), disp_max=("displacement_px", "max"),
        patch_max=("roi_patch_max", "max")).reset_index()
    flags["sat_flag"] = (flags["patch_max"] >= AP["sat_frac"] * AP["sensor_max"]).astype(int)
    qc = pd.DataFrame(qc).merge(post.reset_index(), on="subject_id", how="left")
    log.info(f"QC: gaps={len(gaps)}, stage agreement={stage_agree:.4f}, post-video rows={int(post.sum())}, "
             f"disp-flagged={int(flags.disp_flag.sum())}, saturated ROIs={int(flags.sat_flag.sum())}")
    return R({"qc_subject": qc, "gaps": pd.DataFrame(gaps), "noise": pd.DataFrame(noise), "roi_flags": flags},
             info=dict(stage_agreement=stage_agree, n_gaps=len(gaps), post_video_rows=int(post.sum())))

# ===== SECTION: 16. S4 DARK OFFSET AND ITS UNCERTAINTY =====
def clamp_deltas_under(D, Eb=None, M=None, gains=None, feats=None):
    cam = camera_frames()
    bm = block_meta(cam)
    keep = np.zeros(len(cam), bool)
    for s in cam.subject_id.unique():
        b = bm[bm.subject_id == s].set_index("block")
        ms = (cam.subject_id == s).to_numpy()
        tt = cam["t"].to_numpy()
        keep |= ms & (cam.block.to_numpy() == 0) & (tt >= b.loc[0, "t1"] - AP["baseline_ref_s"])
        for k, bk in b[b.phase == "clamping"].iterrows():
            keep |= ms & (cam.block.to_numpy() == k) & (tt >= bk.t1 - AP["plateau_s"])
    FF = camera_feature_table(cam, D, Eb, M, gains, rows=np.where(keep)[0])
    feats = feats or FEATS
    FF = pd.concat([FF, roi_mean_rows(FF, feats)], ignore_index=True)
    out = []
    for (s, r), g in FF.groupby(["subject_id", "roi_id"]):
        ref = g[g.block == 0][feats].mean()
        for k, gk in g[g.phase == "clamping"].groupby("block"):
            d = gk[feats].mean() - ref
            out.append(dict(subject_id=s, roi_id=r, block=k, pressure=int(gk.pressure.iloc[0]), **d.to_dict()))
    return pd.DataFrame(out)


@step("S4", deps=["S3"], title="Dark offset scenarios")
def s4():
    ch = channel_table()
    D1 = ch["dark_mean"].to_numpy(float)
    rows = []
    for name, mult in AP["dark_scenarios"].items():
        d = clamp_deltas_under(D1 * mult, feats=[AP["primary_feature"]])
        d.insert(0, "scenario", name)
        rows.append(d)
    dd = pd.concat(rows, ignore_index=True)
    f = AP["primary_feature"]
    piv = dd[dd.roi_id == 0].pivot_table(index=["subject_id", "pressure"], columns="scenario", values=f)
    flips = int((np.sign(piv).nunique(axis=1) > 1).sum())
    spread = float((piv.max(1) - piv.min(1)).abs().max())
    log.info(f"Dark scenarios: sign flips in {flips} subject-pressure cells; max spread {spread:.4g}")
    return R({"dark_deltas": dd, "dark_table": ch[["channel_index", "peak_wavelength_nm", "dark_mean", "dark_std"]]},
             arrays={"D": D1}, info=dict(sign_flips=flips, max_spread=spread, primary=AP["primary_dark"],
                                         robust=flips == 0))

# ===== SECTION: 17. S5 SPECTROMETER-TO-CAMERA FORWARD MODEL =====
@step("S5", deps=["S1"], title="Spectrometer-to-camera forward model")
def s5():
    S, meta, grid = load_spectrometer()
    Fm, E, trunc, n_dup = optics_on_grid(grid)
    Wn = band_weights(Fm, E, grid)
    Wm = band_weights(Fm, E, grid, main_lobe_mask(grid))
    Sf = S.astype(np.float64) @ Wn.T
    Sm = S.astype(np.float64) @ Wm.T
    flat = np.ones((1, len(grid))) @ Wn.T
    lin = np.allclose((2 * S[:5].astype(np.float64)) @ Wn.T, 2 * Sf[:5])
    tests = dict(flat_equal=bool(np.allclose(flat, 1.0)), linear=bool(lin), truncation_below_1pct=bool(trunc.max() < 0.01))
    neg = (S < 0)
    support = (Wn.max(0) / Wn.max() > 0.01)
    neg_in_support = int(neg[:, support].sum())
    S_clip = np.clip(S, 0, None).astype(np.float64) @ Wn.T
    Eb, cond = extinction_design(grid, Wn)
    mask = main_lobe_mask(grid)
    share = [(Fm[i] * E * mask[i]).sum() / (Fm[i] * E).sum() for i in range(16)]
    rep = pd.DataFrame(dict(channel=range(16), nm=CH_NM, truncation_share=trunc, mainlobe_share=share,
                            eps_hbo2=Eb[:, 0], eps_hb=Eb[:, 1]))
    if not all(tests.values()):
        raise AssertionError(f"Forward-model unit tests failed: {tests}")
    fig, ax = plt.subplots(figsize=(7.2, 3.0))
    for i in np.argsort(CH_NM):
        ax.plot(grid, Wn[i] / Wn[i].max(), lw=1.0, label=str(CH_NM[i]))
    ax.set(xlabel="Wavelength (nm)", ylabel="Normalised band weight F·E", xlim=(grid.min(), grid.max()))
    ax.legend(ncol=8, fontsize=6, loc="upper center", bbox_to_anchor=(0.5, -0.22))
    savefig(fig, "s05_band_weights", "Effective camera band weights (filter response × xenon irradiance) on the "
            "spectrometer grid.")
    log.info(f"Forward model: max truncation {trunc.max():.4f}; negatives in support {neg_in_support}; "
             f"extinction design condition number {cond:.1f}")
    return R({"spec_meta": meta, "forward_report": rep},
             arrays={"grid": grid, "Wn": Wn, "Wm": Wm, "Sf": Sf, "Sm": Sm, "S_clip": S_clip, "Eb": Eb},
             info=dict(tests=tests, n_negative=int(neg.sum()), negative_in_support=neg_in_support,
                       filter_duplicates=n_dup, mbll_condition=cond, max_truncation=float(trunc.max())))

# ===== SECTION: 18. S6 RELATIVE REFLECTANCE AND EFFECTIVE WHITE =====
@step("S6", deps=["S4", "S5"], title="Relative reflectance and spectrometer-anchored effective gain")
def s6():
    cam = camera_frames()
    D = A("S4", "D")
    refs = baseline_ref_counts(cam)
    Sf, meta = A("S5", "Sf"), T("S5", "spec_meta")
    base = meta.is160.to_numpy() == 0
    gains, rows = {}, []
    for s in subjects_cam():
        cam_base = np.mean([refs[k] for k in refs if k[0] == s], 0) - D
        sp_base = Sf[base & (meta.subject_id == s).to_numpy()].mean(0)
        gains[s] = cam_base / sp_base
        for r in sorted({k[1] for k in refs if k[0] == s}):
            spr = Sf[base & (meta.subject_id == s).to_numpy() & (meta.roi_id == r).to_numpy()].mean(0)
            rows.append(dict(subject_id=s, roi_id=r, **{f"G_{n}": v for n, v in zip(CH_NM, (refs[(s, r)] - D) / spr)}))
    subs = subjects_cam()
    pred = []
    for a, b in itertools.permutations(subs, 2):
        sp_b = Sf[base & (meta.subject_id == b).to_numpy()].mean(0)
        cam_b = np.mean([refs[k] for k in refs if k[0] == b], 0) - D
        err = (gains[a] * sp_b - cam_b) / cam_b
        pred.append(dict(train=a, test=b, **{f"err_{n}": e for n, e in zip(CH_NM, err)}))
    pred = pd.DataFrame(pred)
    ecols = [c for c in pred if c.startswith("err_")]
    n_ok = (pred[ecols].abs() < AP["gain_err_tol"]).sum(1)
    adopt_abs = bool((n_ok >= AP["gain_min_bands"]).all())
    groi = pd.DataFrame(rows)
    gcols = [c for c in groi if c.startswith("G_")]
    cv = (groi.groupby("subject_id")[gcols].std() / groi.groupby("subject_id")[gcols].mean()).reset_index()
    bm = block_meta(cam)
    drift = []
    for (s, r), g in cam[cam.block == 0].groupby(["subject_id", "roi_id"]):
        t, X = g["t"].to_numpy(), g[MEAN_COLS].to_numpy(np.float64) - D
        for i, n in enumerate(CH_NM):
            sl = np.polyfit(t, X[:, i], 1)[0]
            drift.append(dict(subject_id=s, roi_id=r, nm=n, slope_per_100s=sl * 100,
                              rel_drift=sl * 100 / X[:, i].std()))
    log.info(f"Gain model: bands within {AP['gain_err_tol']:.0%} per direction = {n_ok.tolist()}; "
             f"absolute calibration adopted={adopt_abs}")
    return R({"gain_roi": groi, "gain_cv": cv, "cross_prediction": pred, "baseline_drift": pd.DataFrame(drift),
              "gain_subject": pd.DataFrame([dict(subject_id=s, **{f"G_{n}": v for n, v in zip(CH_NM, g)})
                                            for s, g in gains.items()])},
             info=dict(adopt_absolute=adopt_abs, bands_within_tol=n_ok.tolist()))

# ===== SECTION: 19. S7 CROSS-TALK CHARACTERISATION AND CORRECTION =====
@step("S7", deps=["S5", "S6"], title="Cross-talk characterisation and LOPO correction")
def s7():
    Sf, Sm, meta = A("S5", "Sf"), A("S5", "Sm"), T("S5", "spec_meta")
    subs = np.array(sorted(meta.subject_id.unique()))
    sid = meta.subject_id.to_numpy()

    def fit(X, Y, groups):
        best, bs = None, np.inf
        for a in AP["ridge_alphas"]:
            errs = []
            for g in np.unique(groups):
                tr, te = groups != g, groups == g
                m = Ridge(alpha=a, fit_intercept=False).fit(X[tr], Y[tr])
                errs.append(spectral_angle(m.predict(X[te]), Y[te]).mean())
            if np.mean(errs) < bs:
                best, bs = a, np.mean(errs)
        return Ridge(alpha=best, fit_intercept=False).fit(X, Y), best

    folds = []
    for s in subs:
        tr, te = sid != s, sid == s
        m, a = fit(Sf[tr], Sm[tr], sid[tr])
        P = m.predict(Sf[te])
        ang_u, ang_c = spectral_angle(Sf[te], Sm[te]).mean(), spectral_angle(P, Sm[te]).mean()
        rmse_u = np.sqrt(((Sf[te] - Sm[te]) ** 2).mean(0)) / np.abs(Sm[te]).mean(0)
        rmse_c = np.sqrt(((P - Sm[te]) ** 2).mean(0)) / np.abs(Sm[te]).mean(0)
        folds.append(dict(subject_id=s, alpha=a, angle_uncorrected=ang_u, angle_corrected=ang_c,
                          rel_rmse_uncorrected=rmse_u.mean(), rel_rmse_corrected=rmse_c.mean(),
                          improved=bool(ang_c < ang_u)))
    folds = pd.DataFrame(folds)
    final, a_final = fit(Sf, Sm, sid)
    M = final.coef_
    cc = load("color_checker", ["color_index", "color_name", "channel_index", "value", "source_row_index"])
    cc = cc.sort_values("source_row_index")
    cc["rep"] = cc["source_row_index"] // 16
    neutral = []
    for (ci, name), g in cc[cc.color_index >= 19].groupby(["color_index", "color_name"]):
        V = g.pivot_table(index="rep", columns="channel_index", values="value").to_numpy()
        v = V.mean(0)
        vc = v @ M.T
        rep_sd = (V.std(0, ddof=1) / v).mean()
        fb, fa = v.std() / v.mean(), vc.std() / vc.mean()
        neutral.append(dict(color_index=ci, color_name=name, flat_before=fb, flat_after=fa, replicate_cv=rep_sd,
                            passed=bool(fa <= fb + 3 * rep_sd)))
    neutral = pd.DataFrame(neutral)
    n_imp = int(folds.improved.sum())
    adopt = bool(n_imp >= AP["xt_min_folds"] and neutral.passed.all())
    ch = channel_table()
    xt = pd.DataFrame(dict(nm=CH_NM, peak_nm=ch.peak_wavelength_nm, fwhm=ch.fwhm_nm,
                           lobe_ratio=ch.secondary_peak_ratio,
                           out_of_lobe_share=1 - T("S5", "forward_report")["mainlobe_share"].to_numpy()))
    fig, ax = plt.subplots(figsize=(5.2, 3.0))
    x = np.arange(len(folds))
    ax.bar(x - 0.2, folds.angle_uncorrected, 0.4, color="0.6", label="uncorrected")
    ax.bar(x + 0.2, folds.angle_corrected, 0.4, color=C_SUBJ[0], label="corrected (held-out)")
    ax.set(xticks=x, xticklabels=[s[-2:] for s in folds.subject_id], xlabel="Held-out participant",
           ylabel="Spectral angle to main-lobe target (°)")
    ax.legend()
    savefig(fig, "s07_crosstalk_lopo", "Cross-talk correction under leave-one-participant-out: mean spectral angle "
            "between simulated full-filter and main-lobe band values, before and after correction.")
    log.info(f"Cross-talk: improved in {n_imp}/10 folds; neutral patches pass={bool(neutral.passed.all())}; "
             f"adopted as sensitivity variant={adopt}")
    return R({"folds": folds, "neutral_check": neutral, "crosstalk": xt}, arrays={"M": M},
             info=dict(folds_improved=n_imp, neutral_pass=bool(neutral.passed.all()), adopt_variant=adopt,
                       alpha=a_final))

# ===== SECTION: 20. S8 FRAME-LEVEL FEATURES =====
@step("S8", deps=["S3", "S6", "S7"], title="Frame-level features (camera and spectrometer)")
def s8():
    cam = camera_frames()
    D, Eb = A("S4", "D"), A("S5", "Eb")
    cond = I("S5")["mbll_condition"]
    FF = camera_feature_table(cam, D, Eb)
    if cond > AP["mbll_cond_max"]:
        FF[["mbll_hbo2", "mbll_hb", "mbll_diff"]] = np.nan
        log.warning(f"Extinction design ill-conditioned ({cond:.0f}); only mbll_total kept")
    FF = pd.concat([FF, roi_mean_rows(FF, FEATS)], ignore_index=True)
    FF[FEATS] = FF[FEATS].astype(np.float32)
    Sf, meta = A("S5", "Sf"), T("S5", "spec_meta")
    ref = pd.DataFrame(Sf).groupby([meta.subject_id, meta.roi_id, meta.is160]).mean()
    Sref = np.vstack([ref.loc[(s, r, 0)].to_numpy() for s, r in zip(meta.subject_id, meta.roi_id)])
    SF = pd.DataFrame(compute_features(Sf, np.zeros(16), Sref, Eb), columns=FEATS)
    SF = pd.concat([meta.reset_index(drop=True), SF], axis=1)
    rows = rng("s8check").choice(len(cam), size=min(1000, len(cam)), replace=False)
    re = camera_feature_table(cam, D, Eb, rows=np.sort(rows))
    ff_roi = FF[FF.roi_id > 0].set_index(["subject_id", "roi_id", "frame_index"])
    chk = ff_roi.loc[pd.MultiIndex.from_frame(re[["subject_id", "roi_id", "frame_index"]]), FEATS[:-4]].to_numpy()
    ok = bool(np.allclose(chk, re[FEATS[:-4]].to_numpy(), equal_nan=True, rtol=1e-5))
    if not ok:
        raise AssertionError("Feature recomputation check failed")
    dic = pd.DataFrame([dict(feature=f, family=fam) for fam, fs in FAMILIES.items() for f in fs])
    dic["definition"] = dic.family.map({
        "raw": "camera counts (ROI mean), no dark subtraction", "rel": "(I-D)/(I_ref-D), I_ref = last 60 s of baseline",
        "l1": "(I-D)/sum_b(I-D)", "ratio": "(I_a-D_a)/(I_b-D_b), reference 609 or 641 nm",
        "mbll": "least-squares fit of -ln(rel) on filter-weighted Prahl extinction (mM^-1 cm^-1) plus offset; a.u."})
    log.info(f"Frame features: {len(FF):,} rows x {len(FEATS)} features; spectrometer features {SF.shape}")
    return R({"frame_features": FF, "spectro_features": SF, "feature_dictionary": dic},
             info=dict(n_rows=len(FF), n_features=len(FEATS), recompute_check=ok, mbll_condition=cond))

# ===== SECTION: 21. S9 BLOCK FEATURES AND SMALLEST DETECTABLE CHANGE =====
@step("S9", deps=["S8"], title="Block-level features and smallest detectable change")
def s9():
    FF = T("S8", "frame_features")
    cam = camera_frames()
    bm = block_meta(cam)
    bs = block_stats(FF, bm, FEATS)
    dl = deltas(bs, FEATS)
    kin = kinetics(FF, bm, bs, AP["key_features"])
    sdc_rows = []
    w = AP["sdc_win_s"]
    for (s, r), g in FF[FF.block == 0].groupby(["subject_id", "roi_id"]):
        _, Bm = bins_1s(g["t"].to_numpy(), g[FEATS].to_numpy(np.float64), g["t"].min())
        sd = sdc_from_bins(Bm, w)
        rr = rng(f"sdc{s}{r}")
        boots = np.vstack([sdc_from_bins(Bm[mbb_index(len(Bm), AP["sdc_block_s"], rr)], w)
                           for _ in range(AP["sdc_boot"])])
        pl = dl[(dl.subject_id == s) & (dl.roi_id == r) & (dl.contrast == "placebo")][FEATS].abs()
        for stat, v in [("sdc95", sd), ("sdc95_mbb_median", np.nanmedian(boots, 0)),
                        ("sdc95_mbb_lo", np.nanpercentile(boots, 2.5, 0)),
                        ("sdc95_mbb_hi", np.nanpercentile(boots, 97.5, 0)),
                        ("sdc95_rest", pl.quantile(0.95).to_numpy())]:
            sdc_rows.append(dict(subject_id=s, roi_id=r, stat=stat, **dict(zip(FEATS, v))))
    sdc = pd.DataFrame(sdc_rows)
    n_invalid = int((~bs["valid"]).sum())
    log.info(f"Block stats: {len(bs)} windows ({n_invalid} invalid); deltas {len(dl)}; kinetics {len(kin)}")
    return R({"block_stats": bs, "deltas": dl, "kinetics": kin, "sdc": sdc, "block_meta": bm},
             info=dict(invalid_windows=n_invalid))

# ===== SECTION: 22. S10 DOSE–RESPONSE AND HYPOTHESIS H1 =====
@step("S10", deps=["S9"], title="Dose-response and H1 (camera arm + spectrometer arm)", critical=False)
def s10():
    f = AP["primary_feature"]
    dl, sdc, FF, bm = T("S9", "deltas"), T("S9", "sdc"), T("S8", "frame_features"), T("S9", "block_meta")
    cam_rows, sec = [], []
    for s in sorted(dl.subject_id.unique()):
        d = dl[(dl.subject_id == s) & (dl.roi_id == 0)]
        loc = d[d.contrast == "local"].sort_values("pressure")
        plc = d[d.contrast == "placebo"].sort_values("pressure")
        x, y, yp = loc.pressure.to_numpy(float), loc[f].to_numpy(float), plc[f].to_numpy(float)
        sd = float(sdc[(sdc.subject_id == s) & (sdc.roi_id == 0) & (sdc.stat == "sdc95")][f].iloc[0])
        rho, p = perm_spearman(x, y, "less")
        onset = next((int(p_) for p_ in x if np.all(y[x >= p_] < -sd)), None)
        neg60 = bool(np.all(y[x >= AP["h1_min_pressure"]] < -sd))
        tsc, tsp = theil(y, x), theil(yp, plc.pressure.to_numpy(float))
        drift_ok = bool(tsc < tsp and abs(tsc) > abs(tsp))
        roi_cons = []
        for r in sorted(dl[(dl.subject_id == s) & (dl.roi_id > 0)].roi_id.unique()):
            yr = dl[(dl.subject_id == s) & (dl.roi_id == r) & (dl.contrast == "local")].sort_values("pressure")
            roi_cons.append(bool((yr[yr.pressure >= AP["h1_min_pressure"]][f] < 0).all()))
        cam_rows.append(dict(subject_id=s, rho=rho, p_trend=p, sdc95=sd, onset_mmhg=onset, all_below_sdc_ge60=neg60,
                             theil_clamp=tsc, theil_placebo=tsp, clamp_exceeds_placebo=drift_ok,
                             rois_all_negative_ge60=f"{sum(roi_cons)}/{len(roi_cons)}",
                             exploratory_peak_mmhg=int(x[np.nanargmin(y)]), exploratory_peak_delta=float(np.nanmin(y)),
                             supported=bool(p < AP["alpha"] and neg60 and drift_ok),
                             **{f"d_{int(a)}": b for a, b in zip(x, y)}))
        for fam, cols in FAMILIES.items():
            ps = [perm_spearman(x, loc[c].to_numpy(float))[1] for c in cols]
            rh = [perm_spearman(x, loc[c].to_numpy(float))[0] for c in cols]
            for c, rr_, pp, ph in zip(cols, rh, ps, holm(ps)):
                sec.append(dict(arm="camera", subject_id=s, family=fam, feature=c, rho=rr_, p=pp, p_holm=ph))
    cam_df = pd.DataFrame(cam_rows)
    mm = None
    if SM is not None:
        try:
            dd = dl[(dl.roi_id > 0) & (dl.contrast == "local")][["subject_id", "roi_id", "pressure", f]].dropna()
            dd = dd.rename(columns={f: "y"})
            dd["g"] = dd.subject_id + "_" + dd.roi_id.astype(str)
            res = smf.mixedlm("y ~ C(pressure) + C(subject_id)", dd, groups=dd["g"]).fit(reml=True)
            mm = pd.DataFrame({"term": res.params.index, "estimate": res.params.values, "se": res.bse.values,
                               "p": res.pvalues.values})
        except Exception as e:
            log.error(f"S10 descriptive mixed model failed: {e!r}\n{traceback.format_exc()}")
    SF = T("S8", "spectro_features")
    Sf = A("S5", "Sf")
    lr = np.log(Sf[:, I550] / Sf[:, I609])
    sp = pd.DataFrame({"s": SF.subject_id, "r": SF.roi_id, "k": SF.is160, "lr": lr})
    per = sp.groupby(["s", "r", "k"])["lr"].mean().groupby(["s", "k"]).mean().unstack()
    diff = (per[1] - per[0]).to_numpy()
    p_w = wilcoxon_exact(diff, "less")
    ci = boot_ci(diff, tag="s10spec")
    spec_supported = bool(p_w < AP["alpha"] and ci[1] < 0)
    lmm = None
    if SM is not None:
        try:
            res = smf.mixedlm("lr ~ k", sp, groups=sp["s"], re_formula="~k",
                              vc_formula={"roi": "0 + C(r)"}).fit(reml=True)
            lmm = dict(model="random intercept + random stage slope per participant; ROI variance component",
                       estimate=float(res.params["k"]), se=float(res.bse["k"]), p=float(res.pvalues["k"]),
                       converged=bool(res.converged))
        except Exception as e:
            log.error(f"S10 spectrometer mixed model failed: {e!r}\n{traceback.format_exc()}")
    for fam, cols in {**SPEC_FAMILIES, "rel": REL, "mbll": MBLL}.items():
        sub = SF.groupby(["subject_id", "roi_id", "is160"])[cols].mean().groupby(["subject_id", "is160"]).mean()
        dd = sub.xs(1, level=1) - sub.xs(0, level=1)
        ps = [wilcoxon_exact(dd[c].to_numpy(), "two-sided") for c in cols]
        for c, pp, ph in zip(cols, ps, holm(ps)):
            sec.append(dict(arm="spectrometer", subject_id="all", family=fam, feature=c, rho=float(dd[c].mean()),
                            p=pp, p_holm=ph))
    cam_sup = int(cam_df.supported.sum())
    h1 = "supported" if (cam_sup == len(cam_df) and spec_supported) else (
        "partially supported" if (cam_sup == len(cam_df) or spec_supported) else "not supported")
    spec_tab = pd.DataFrame({"subject_id": per.index, "baseline": per[0].values, "cuff160": per[1].values,
                             "diff": diff})
    # figures
    fig, axes = plt.subplots(len(cam_df), 1, figsize=(7.2, 2.3 * len(cam_df)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, s, c in zip(axes, cam_df.subject_id, C_SUBJ):
        g = FF[(FF.subject_id == s) & (FF.roi_id == 0)].sort_values("t")
        base = g[g.block == 0][f].mean()
        y = (g[f] / base - 1) * 100
        yy = pd.Series(y.to_numpy(), index=pd.to_timedelta(g.t.to_numpy(), unit="s")).rolling("2s").median().to_numpy()
        tt = g.t.to_numpy()
        yy = np.where(np.r_[False, np.diff(tt) > AP["gap_s"]], np.nan, yy)
        shade_clamps(ax, bm, s)
        ax.plot(tt, yy, color=c, lw=0.8)
        ax.axhline(0, color="0.3", lw=0.6)
        ax.set_ylabel(f"{s[-2:]}: Δ 550/609 (%)")
    axes[-1].set_xlabel("Time (s)")
    savefig(fig, "s10_ratio_timecourse", "ROI-mean 550/609 nm ratio relative to baseline (2 s rolling median); "
            "shading = cuff inflation with pressure (mmHg).")
    fig, ax = plt.subplots(figsize=(5.4, 3.3))
    for (_, rw), c in zip(cam_df.iterrows(), C_SUBJ):
        loc = dl[(dl.subject_id == rw.subject_id) & (dl.roi_id == 0) & (dl.contrast == "local")].sort_values("pressure")
        plc = dl[(dl.subject_id == rw.subject_id) & (dl.roi_id == 0) & (dl.contrast == "placebo")].sort_values("pressure")
        ax.plot(loc.pressure, loc[f], "o-", color=c, ms=4, label=f"{rw.subject_id[-2:]} clamp (local Δ)")
        ax.plot(plc.pressure, plc[f], "x--", color=c, ms=4, lw=0.8, alpha=0.7, label=f"{rw.subject_id[-2:]} placebo")
        ax.axhspan(-rw.sdc95, rw.sdc95, color=c, alpha=0.08)
    ax.axhline(0, color="0.3", lw=0.6)
    ax.set(xlabel="Cuff pressure (mmHg)", ylabel="Δ 550/609 ratio")
    ax.legend(fontsize=7, ncol=2)
    savefig(fig, "s10_dose_response", "Local change of the ROI-mean 550/609 ratio per clamping stage, placebo "
            "differences between consecutive rest references, and ±SDC95 bands (shaded).")
    fig, ax = plt.subplots(figsize=(3.6, 3.3))
    for i, rw in spec_tab.iterrows():
        ax.plot([0, 1], [rw.baseline, rw.cuff160], "-o", color="0.55", ms=3, lw=0.8)
    ax.plot([0, 1], [spec_tab.baseline.mean(), spec_tab.cuff160.mean()], "-o", color=C_SUBJ[0], lw=2)
    ax.set(xticks=[0, 1], xticklabels=["baseline", "160 mmHg"], ylabel="log(S550 / S609), spectrometer",
           xlim=(-0.3, 1.3))
    savefig(fig, "s10_spectrometer_arm", "Spectrometer arm of H1: participant means of the simulated camera-band "
            "log ratio at baseline and 160 mmHg (n = 10); blue = group mean.")
    log.info(f"H1: camera supported in {cam_sup}/{len(cam_df)}; spectrometer p={p_w:.4g}, CI={ci}; overall={h1}")
    log.info("Exploratory (not pre-specified): pressure of largest local decrease per participant: "
             + ", ".join(f"{r.subject_id}={r.exploratory_peak_mmhg} mmHg" for r in cam_df.itertuples()))
    tabs = {"h1_camera": cam_df, "h1_spectrometer": spec_tab, "secondary": pd.DataFrame(sec)}
    if mm is not None:
        tabs["mixed_model_camera"] = mm
    return R(tabs, info=dict(h1=h1, camera_supported=cam_sup, spectro_p=p_w, spectro_ci=ci,
                             spectro_mean_diff=float(np.mean(diff)), spectro_supported=spec_supported,
                             spectro_lmm=lmm, signed_off=PLAN_SIGNED_OFF,
                             note="camera arm non-confirmatory (hypothesis-generating data)"))

# ===== SECTION: 23. S11 RECOVERY KINETICS (RQ2) =====
def expfit(t, y):
    f = lambda tt, a, b, tau: a + b * np.exp(-tt / tau)
    ok = np.isfinite(y)
    t, y = t[ok], y[ok]
    if len(t) < 8:
        return None
    p0 = [np.mean(y[-5:]), y[0] - np.mean(y[-5:]), 10.0]
    lo, hi = AP["tau_bounds"]
    p, _ = optimize.curve_fit(f, t, y, p0=p0, bounds=([-np.inf, -np.inf, lo], [np.inf, np.inf, hi]), maxfev=20000)
    yh = f(t, *p)
    r2 = 1 - ((y - yh) ** 2).sum() / max(((y - y.mean()) ** 2).sum(), 1e-30)
    return p, r2, yh, t, y


@step("S11", deps=["S9"], title="Recovery kinetics (RQ2)", critical=False)
def s11():
    FF, bs, bm = T("S8", "frame_features"), T("S9", "block_stats"), T("S9", "block_meta")
    kin = T("S9", "kinetics")
    sdc = T("S9", "sdc")
    sdc95 = sdc[sdc.stat == "sdc95"].set_index(["subject_id", "roi_id"])
    rows, examples = [], {}
    for (s, r), g in FF.groupby(["subject_id", "roi_id"]):
        gi = bs[(bs.subject_id == s) & (bs.roi_id == r)].set_index(["block", "window"])
        for k in range(2, 17, 2):
            bk = bm[(bm.subject_id == s) & (bm.block == k)].iloc[0]
            gg = g[g.block == k]
            for f in AP["key_features"]:
                tt, yy = bins_1s(gg.t.to_numpy() - bk.t0, gg[[f]].to_numpy(np.float64), 0.0)
                tt, yy = tt.astype(float), yy[:, 0]
                kc = k - 1
                ref = gi.loc[(kc - 1, "rest_ref"), f] if kc - 1 >= 2 else gi.loc[(0, "ref_global"), f]
                pl = gi.loc[(kc, "plateau"), f]
                rec = dict(subject_id=s, roi_id=r, block=k, prev_pressure=int(bk.prev_pressure), feature=f)
                try:
                    out = expfit(tt, yy)
                except Exception as e:
                    out = None
                    rec["fit_error"] = repr(e)
                if out is not None:
                    p, r2, yh, t_, y_ = out
                    res = y_ - yh
                    rr = rng(f"kin{s}{r}{k}{f}")
                    taus = []
                    for _ in range(AP["kin_boot"]):
                        yb = yh + res[mbb_index(len(res), 5, rr)]
                        try:
                            taus.append(expfit(t_, yb)[0][2])
                        except Exception:
                            taus.append(np.nan)
                    tau_hi = np.nanpercentile(taus, 97.5) if np.isfinite(taus).any() else np.nan
                    rec.update(a=p[0], b=p[1], tau=p[2], r2=r2, tau_hi=tau_hi,
                               accepted=bool(r2 >= AP["fit_r2_min"] and tau_hi < AP["tau_bounds"][1]))
                    if f == AP["primary_feature"] and r == 0:
                        examples[(s, k)] = (t_, y_, yh)
                else:
                    rec.update(tau=np.nan, r2=np.nan, tau_hi=np.nan, accepted=False)
                rec["clamp_change_exceeds_sdc"] = bool(abs(pl - ref) > sdc95.loc[(s, r), f])
                if rec["clamp_change_exceeds_sdc"]:
                    frac = (yy - pl) / (ref - pl)
                    i20 = np.argmin(np.abs(tt - AP["recovery_t"]))
                    rec["recovery_frac_20s"] = float(frac[i20]) if len(frac) else np.nan
                    hit = np.where(frac >= 0.5)[0]
                    rec["t_half"] = float(tt[hit[0]]) if len(hit) else np.nan
                else:
                    rec["recovery_frac_20s"], rec["t_half"] = np.nan, np.nan
                rec["overshoot"] = kin[(kin.subject_id == s) & (kin.roi_id == r) & (kin.block == k) &
                                       (kin.feature == f)]["overshoot"].iloc[0]
                rows.append(rec)
    kt = pd.DataFrame(rows)
    f = AP["primary_feature"]
    prim = kt[(kt.feature == f) & (kt.roi_id > 0)]
    share = float(prim.accepted.mean())
    use_model = share >= AP["fit_accept_share"]
    tests = []
    for s in sorted(kt.subject_id.unique()):
        d = kt[(kt.subject_id == s) & (kt.roi_id == 0) & (kt.feature == f) & (kt.prev_pressure >= AP["rq2_min_pressure"])]
        metrics = (["tau"] if use_model else []) + ["t_half", "overshoot"]
        ps, rhos = [], []
        for m in metrics:
            dm = d[d.accepted] if m == "tau" else d
            rho, p = perm_spearman(dm.prev_pressure.to_numpy(float), dm[m].to_numpy(float))
            ps.append(p)
            rhos.append(rho)
        for m, rho, p, ph in zip(metrics, rhos, ps, holm(ps)):
            tests.append(dict(subject_id=s, metric=m, rho=rho, p=p, p_holm=ph))
    tests = pd.DataFrame(tests)
    dep = tests.groupby("metric").apply(lambda g: bool((g.p_holm < AP["alpha"]).all() and len(g) == 2),
                                        include_groups=False).to_dict() if len(tests) else {}
    incomplete = kt[(kt.feature == f) & (kt.roi_id == 0)].groupby("subject_id")["recovery_frac_20s"].apply(
        lambda x: f"{int((x < 1).sum())}/{int(x.notna().sum())} evaluable blocks").to_dict()
    if examples:
        s0_ = sorted({k[0] for k in examples})[0]
        ks = sorted(k for (ss, k) in examples if ss == s0_)
        fig, axes = plt.subplots(2, 4, figsize=(8.4, 3.8), sharey=True)
        for ax, k in zip(axes.ravel(), ks):
            t_, y_, yh = examples[(s0_, k)]
            ax.plot(t_, y_, ".", ms=2, color="0.5")
            ax.plot(t_, yh, color=C_SUBJ[0], lw=1.2)
            pp = kt[(kt.subject_id == s0_) & (kt.roi_id == 0) & (kt.block == k) & (kt.feature == f)].iloc[0]
            ax.set_title(f"after {pp.prev_pressure} mmHg, τ={pp.tau:.1f}s", fontsize=7.5)
        fig.supxlabel("Time since release (s)")
        fig.supylabel("550/609 ratio")
        savefig(fig, "s11_recovery_fits", f"Mono-exponential recovery fits in the eight rest blocks ({s0_}, ROI mean).")
    log.info(f"RQ2: accepted fit share {share:.2f} (model-based={use_model}); dependence={dep}")
    return R({"kinetics_fits": kt, "rq2_tests": tests},
             info=dict(accepted_share=share, model_based=use_model, dependence=dep, incomplete_recovery=incomplete))

# ===== SECTION: 24. S12 FEATURE ROBUSTNESS (RQ3) =====
@step("S12", deps=["S9", "S4", "S7"], title="Feature-family robustness (RQ3)", critical=False)
def s12():
    FF, sdc, dl = T("S8", "frame_features"), T("S9", "sdc"), T("S9", "deltas")
    SF = T("S8", "spectro_features")
    D, Eb = A("S4", "D"), A("S5", "Eb")
    rows = []
    base = SF[SF.is160 == 0].groupby(["subject_id", "roi_id"])[FEATS].mean().groupby("subject_id").mean()
    c1 = (base.std() / base.mean().abs())
    sp = SF.groupby(["subject_id", "roi_id", "is160"])[FEATS].mean().groupby(["subject_id", "is160"]).mean()
    sd_ = sp.xs(1, level=1) - sp.xs(0, level=1)
    dz = sd_.mean() / sd_.std()
    drift = []
    for (s, r), g in FF[(FF.block == 0) & (FF.roi_id > 0)].groupby(["subject_id", "roi_id"]):
        t = g.t.to_numpy()
        X = g[FEATS].to_numpy(np.float64)
        with np.errstate(invalid="ignore", divide="ignore"):
            sl = np.array([np.polyfit(t, X[:, i], 1)[0] if np.isfinite(X[:, i]).all() else np.nan
                           for i in range(len(FEATS))])
            drift.append(np.abs(sl * 100 / np.nanstd(X, 0)))
    c2 = pd.Series(np.nanmedian(np.vstack(drift), 0), index=FEATS)
    d160 = dl[(dl.roi_id == 0) & (dl.contrast == "global") & (dl.pressure == 160)].set_index("subject_id")[FEATS]
    sd0 = sdc[(sdc.roi_id == 0) & (sdc.stat == "sdc95")].set_index("subject_id")[FEATS]
    c3 = (d160.abs() / sd0).mean()
    sens = {}
    for name, mult in AP["dark_scenarios"].items():
        d = clamp_deltas_under(D * mult, Eb)
        sens[name] = d[(d.roi_id == 0) & (d.pressure == 160)].set_index("subject_id")[FEATS]
    variants = [n for n in sens if n != AP["primary_dark"]]
    if I("S7")["adopt_variant"]:
        G = T("S6", "gain_subject").set_index("subject_id")
        gains = {s: G.loc[s].to_numpy(float) for s in G.index}
        d = clamp_deltas_under(D, Eb, A("S7", "M"), gains)
        sens["xtalk"] = d[(d.roi_id == 0) & (d.pressure == 160)].set_index("subject_id")[FEATS]
        variants.append("xtalk")
    ref = sens[AP["primary_dark"]]
    c4 = pd.concat([((sens[v] - ref).abs() / ref.abs()).mean() for v in variants], axis=1).max(1)
    per = pd.DataFrame({"between_participant_cv": c1, "baseline_drift": c2, "effect_to_noise_camera": c3,
                        "spectro_dz_abs": dz.abs(), "processing_sensitivity": c4})
    per["family"] = [next(k for k, v in FAMILIES.items() if f in v) for f in per.index]
    per.loc[per.family.isin(["rel", "mbll"]), "between_participant_cv"] = np.nan
    per.loc[per.family == "raw", "processing_sensitivity"] = np.nan
    fam = per.groupby("family").median(numeric_only=True)
    better = {"between_participant_cv": -1, "baseline_drift": -1, "effect_to_noise_camera": 1, "processing_sensitivity": -1}
    verdict = []
    for fm in fam.index:
        if fm == "raw":
            continue
        nb, nw = 0, 0
        for c, sgn in better.items():
            a, b = fam.loc[fm, c], fam.loc["raw", c]
            if not (np.isfinite(a) and np.isfinite(b)):
                continue
            if sgn * (a - b) > 0:
                nb += 1
            elif sgn * (a - b) < 0:
                nw += 1
        verdict.append(dict(family=fm, better_than_raw=nb, worse_than_raw=nw, robust=bool(nw == 0 and nb >= 2)))
    verdict = pd.DataFrame(verdict)
    log.info("RQ3 robust families: " + ", ".join(verdict[verdict.robust].family) if verdict.robust.any()
             else "RQ3: no family met the robustness rule")
    note = ("processing_sensitivity is undefined for raw counts (no dark subtraction is applied to them) and is "
            "excluded from comparisons with raw; between_participant_cv is undefined for baseline-referenced families "
            "(rel, mbll). Implementation note added after the first test run, before any confirmatory run.")
    return R({"feature_robustness": per.reset_index(names="feature"), "family_robustness": fam.reset_index(),
              "rq3_verdict": verdict}, info=dict(robust_families=verdict[verdict.robust].family.tolist(),
                                                 variants=variants, note=note))

# ===== SECTION: 25. S13 CAMERA–SPECTROMETER AGREEMENT AND REPEATABILITY =====
@step("S13", deps=["S6", "S9"], title="Camera-spectrometer agreement and repeatability (O5, T2)", critical=False)
def s13():
    cam = camera_frames()
    D = A("S4", "D")
    bm = T("S9", "block_meta")
    Sf, meta = A("S5", "Sf"), T("S5", "spec_meta")
    refs = baseline_ref_counts(cam)
    units = []
    for (s, r), g in cam.groupby(["subject_id", "roi_id"]):
        k160 = int(bm[(bm.subject_id == s) & (bm.pressure == 160)].block.iloc[0])
        t1 = bm[(bm.subject_id == s) & (bm.block == k160)].t1.iloc[0]
        g160 = g[g.block == k160]
        xb = refs[(s, r)] - D
        xc = g160[MEAN_COLS].to_numpy(np.float64).mean(0) - D
        xl = g160[g160.t >= t1 - 30][MEAN_COLS].to_numpy(np.float64).mean(0) - D
        ms = (meta.subject_id == s).to_numpy() & (meta.roi_id == r).to_numpy()
        sp = Sf[ms & (meta.is160 == 1).to_numpy()].mean(0) / Sf[ms & (meta.is160 == 0).to_numpy()].mean(0)
        units.append(dict(subject_id=s, roi_id=r, cam=100 * (xc / xb - 1), cam_last30=100 * (xl / xb - 1),
                          spec=100 * (sp - 1)))
    roi_u = units
    part_u = []
    for s in sorted({u["subject_id"] for u in units}):
        us = [u for u in units if u["subject_id"] == s]
        ms = (meta.subject_id == s).to_numpy()
        sp = Sf[ms & (meta.is160 == 1).to_numpy()].mean(0) / Sf[ms & (meta.is160 == 0).to_numpy()].mean(0)
        part_u.append(dict(subject_id=s, roi_id=0, cam=np.mean([u["cam"] for u in us], 0),
                           cam_last30=np.mean([u["cam_last30"] for u in us], 0), spec=100 * (sp - 1)))
    out_ba, out_sum = [], []
    for level, U in [("roi", roi_u), ("participant", part_u)]:
        for cam_key in ["cam", "cam_last30"]:
            C = np.vstack([u[cam_key] for u in U])
            Sp = np.vstack([u["spec"] for u in U])
            for i, nm in enumerate(CH_NM):
                ba = bland_altman(C[:, i], Sp[:, i])
                ci = boot_ci(C[:, i] - Sp[:, i], tag=f"ba{level}{cam_key}{i}") if len(U) > 2 else (np.nan, np.nan)
                out_ba.append(dict(level=level, camera_window=cam_key, nm=nm, **ba, bias_ci_lo=ci[0], bias_ci_hi=ci[1],
                                   icc21=icc21(np.c_[C[:, i], Sp[:, i]])))
            d = (C - Sp).ravel()
            grp = np.repeat(np.arange(len(U)), 16)
            pooled = bland_altman_repeated(d, grp) if len(U) > 1 else {}
            ang = spectral_angle(C, Sp)
            out_sum.append(dict(level=level, camera_window=cam_key, n_units=len(U), **{f"pooled_{k}": v for k, v in pooled.items()},
                                icc21_pooled=icc21(np.c_[C.ravel(), Sp.ravel()]), angle_mean=float(ang.mean()),
                                angle_ci=boot_ci(ang, tag=f"ang{level}{cam_key}") if len(U) > 2 else (np.nan, np.nan)))
    ba_df, sum_df = pd.DataFrame(out_ba), pd.DataFrame(out_sum)
    primary_level = "roi" if ROI_CORRESPONDENCE_CONFIRMED else "participant"
    abs_tab = None
    if I("S6")["adopt_absolute"]:
        G = T("S6", "gain_subject").set_index("subject_id")
        subs = list(G.index)
        rows = []
        for a, b in itertools.permutations(subs, 2):
            for k in (0, 1):
                ms = (meta.subject_id == b).to_numpy() & (meta.is160 == k).to_numpy()
                kk = 0 if k == 0 else int(bm[(bm.subject_id == b) & (bm.pressure == 160)].block.iloc[0])
                Xc = np.mean([cam[(cam.subject_id == b) & (cam.roi_id == r) & (cam.block == kk)][MEAN_COLS].to_numpy().mean(0)
                              for r in cam[cam.subject_id == b].roi_id.unique()], 0) - D
                Rc = Xc / G.loc[a].to_numpy(float)
                rows.append(dict(gain_from=a, subject_id=b, stage=["baseline", "160"][k],
                                 **{f"d_{n}": v for n, v in zip(CH_NM, 100 * (Rc / Sf[ms].mean(0) - 1))}))
        abs_tab = pd.DataFrame(rows)
    rep = []
    for (s, r, k), idx in meta.groupby(["subject_id", "roi_id", "is160"]).groups.items():
        X = Sf[np.asarray(list(idx))]
        rep.append(dict(subject_id=s, roi_id=r, is160=k, **{f"cv_{n}": v for n, v in zip(CH_NM, X.std(0, ddof=1) / X.mean(0))}))
    rep = pd.DataFrame(rep)
    roi_means = pd.DataFrame(Sf).groupby([meta.subject_id, meta.is160, meta.roi_id]).mean()
    between = []
    for (s, k), g in roi_means.groupby(level=[0, 1]):
        X = g.to_numpy()
        between.append(dict(subject_id=s, is160=k, **{f"cv_{n}": v for n, v in zip(CH_NM, X.std(0, ddof=1) / X.mean(0))}))
    between = pd.DataFrame(between)
    iccs = []
    for i, nm in enumerate(CH_NM):
        Mx = roi_means.xs(0, level=1)[i].unstack("roi_id").to_numpy()
        iccs.append(dict(nm=nm, icc21_across_rois_baseline=icc21(Mx)))
    cam_rep = []
    for s, g in cam[cam.block == 0].groupby("subject_id"):
        X = np.vstack([refs[k] - D for k in refs if k[0] == s])
        cam_rep.append(dict(subject_id=s, n_rois=len(X), **{f"cv_{n}": v for n, v in zip(CH_NM, X.std(0, ddof=1) / X.mean(0))}))
    dl = T("S9", "deltas")
    f = AP["primary_feature"]
    cam_delta_rep = dl[(dl.roi_id > 0) & (dl.contrast == "global")].groupby(["subject_id", "pressure"])[f].agg(
        ["mean", "std", "count"]).reset_index()
    ba_p = ba_df[(ba_df.level == primary_level) & (ba_df.camera_window == "cam")]
    fig, ax = plt.subplots(figsize=(5.0, 3.4))
    U = roi_u if primary_level == "roi" else part_u
    C = np.vstack([u["cam"] for u in U])
    Sp = np.vstack([u["spec"] for u in U])
    mean_, diff_ = ((C + Sp) / 2).ravel(), (C - Sp).ravel()
    ax.scatter(mean_, diff_, s=10, c=np.tile(np.arange(16), len(U)), cmap="viridis")
    sm_ = sum_df[(sum_df.level == primary_level) & (sum_df.camera_window == "cam")].iloc[0]
    for v, ls in [(sm_.get("pooled_bias"), "-"), (sm_.get("pooled_loa_lo"), "--"), (sm_.get("pooled_loa_hi"), "--")]:
        if v is not None and np.isfinite(v):
            ax.axhline(v, color="0.3", ls=ls, lw=0.9)
    ax.set(xlabel="Mean of camera and spectrometer change, 160 mmHg vs baseline (%)",
           ylabel="Camera − spectrometer (%)")
    savefig(fig, "s13_bland_altman", f"Bland–Altman plot of per-band relative change at 160 mmHg ({primary_level} "
            "level; colour = band); lines = pooled bias and 95% limits of agreement (repeated-measures method).")
    log.info(f"Agreement ({primary_level} level): pooled bias {sm_.get('pooled_bias', np.nan):.3g}%, "
             f"LoA {sm_.get('pooled_loa_lo', np.nan):.3g}..{sm_.get('pooled_loa_hi', np.nan):.3g}%")
    tabs = {"ba_per_band": ba_df, "agreement_summary": sum_df,
            "spectro_replicate_cv": rep, "spectro_between_roi_cv": between, "spectro_icc_rois": pd.DataFrame(iccs),
            "camera_between_roi_cv": pd.DataFrame(cam_rep), "camera_delta_repeatability": cam_delta_rep}
    if abs_tab is not None:
        tabs["absolute_agreement"] = abs_tab
    return R(tabs, info=dict(primary_level=primary_level, roi_correspondence_confirmed=ROI_CORRESPONDENCE_CONFIRMED,
                             absolute_used=abs_tab is not None,
                             note="ROI-level results are conditional on ROI correspondence"
                             if not ROI_CORRESPONDENCE_CONFIRMED else ""))

# ===== SECTION: 26. MODELS: CLASSICAL LADDER AND TEMPORAL BACKBONES =====
class PLSDA(BaseEstimator, ClassifierMixin):
    def __init__(self, n=2):
        self.n = n

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        Y = (y[:, None] == self.classes_[None, :]).astype(float)
        self.m_ = PLSRegression(n_components=min(self.n, X.shape[1]), scale=False).fit(X, Y)
        return self

    def predict(self, X):
        return self.classes_[np.argmax(self.m_.predict(X), 1)]


class PLSR(BaseEstimator):
    def __init__(self, n=2):
        self.n = n

    def fit(self, X, y):
        self.m_ = PLSRegression(n_components=min(self.n, X.shape[1]), scale=False).fit(X, y)
        return self

    def predict(self, X):
        return np.ravel(self.m_.predict(X))


def make_model(name, task, hp):
    cls = task == "cls"
    pre = [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]
    if name == "majority":
        return DummyClassifier(strategy="most_frequent") if cls else DummyRegressor(strategy="mean")
    if name == "logreg_pca":
        est = LogisticRegression(C=hp["C"], class_weight="balanced", max_iter=3000) if cls else Ridge(alpha=1.0 / hp["C"])
        return Pipeline(pre + [("pca", PCA(n_components=0.95, svd_solver="full")), ("m", est)])
    if name == "pls":
        return Pipeline(pre + [("m", PLSDA(hp["n"]) if cls else PLSR(hp["n"]))])
    if name in ("rf", "time_only"):
        kw = dict(n_estimators=RF_TREES, max_depth=hp.get("max_depth"), min_samples_leaf=5, n_jobs=N_JOBS,
                  random_state=SEED)
        est = RandomForestClassifier(class_weight="balanced_subsample", **kw) if cls else RandomForestRegressor(**kw)
        return Pipeline(pre[:1] + [("m", est)])
    if name == "svm":
        est = SVC(C=hp["C"], gamma="scale", class_weight="balanced") if cls else SVR(C=hp["C"], gamma="scale")
        return Pipeline(pre + [("m", est)])
    raise ValueError(name)


def score(task, y, p):
    return balanced_accuracy_score(y, p) if task == "cls" else -np.sqrt(mean_squared_error(y, p))


def metrics(task, y, p, pressure=None):
    if task == "cls":
        labels = [0, 1, 2]
        cm = confusion_matrix(y, p, labels=labels)
        sens = np.diag(cm) / np.maximum(cm.sum(1), 1)
        spec = [(cm.sum() - cm[i].sum() - cm[:, i].sum() + cm[i, i]) / max(cm.sum() - cm[i].sum(), 1) for i in labels]
        return dict(bal_acc=balanced_accuracy_score(y, p), macro_f1=f1_score(y, p, average="macro", labels=labels,
                    zero_division=0), sens=sens.tolist(), spec=spec, confusion=cm.tolist())
    out = dict(rmse=float(np.sqrt(mean_squared_error(y, p))), mae=float(mean_absolute_error(y, p)),
               r2=float(r2_score(y, p)))
    if pressure is not None:
        out["rmse_by_pressure"] = {int(k): float(np.sqrt(np.mean((p[pressure == k] - y[pressure == k]) ** 2)))
                                   for k in np.unique(pressure)}
    return out


def mbb_metric_ci(task, y, p, order_key, B=None, tag="ci"):
    idx = np.argsort(order_key, kind="stable")
    y, p = y[idx], p[idx]
    r = rng(tag)
    vals = []
    for _ in range(B or min(AP["boot"], 1000)):
        j = mbb_index(len(y), AP["mbb_block"], r)
        vals.append(score(task, y[j], p[j]) if task == "cls" else -score(task, y[j], p[j]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


if TORCH is not None:
    class TCN(nn.Module):
        def __init__(self, k, h, out, drop):
            super().__init__()
            layers, c = [], k
            for d in (1, 2, 4):
                layers += [nn.Conv1d(c, h, 5, padding=2 * d, dilation=d), nn.ReLU(), nn.Dropout(drop)]
                c = h
            self.net, self.head = nn.Sequential(*layers), nn.Linear(h, out)

        def forward(self, x):
            return self.head(self.net(x.transpose(1, 2)).mean(-1))

    class GRUNet(nn.Module):
        def __init__(self, k, h, out, drop):
            super().__init__()
            self.gru, self.drop, self.head = nn.GRU(k, h, batch_first=True), nn.Dropout(drop), nn.Linear(h, out)

        def forward(self, x):
            _, hN = self.gru(x)
            return self.head(self.drop(hN[-1]))

    BACKBONES = {"tcn": TCN, "gru": GRUNet}

# ===== SECTION: 27. WINDOWS, SPLITS AND EXPERIMENT RUNNERS =====
def build_windows(FF, bm):
    F = FF[FF.roi_id > 0].sort_values(["subject_id", "roi_id", "t"]).reset_index(drop=True)
    cols = FEATS
    Xall = F[cols].to_numpy(np.float64)
    recs, mean_l, slope_l = [], [], []
    for (s, r), g in F.groupby(["subject_id", "roi_id"]):
        ii = g.index.to_numpy()
        t = g.t.to_numpy()
        fps = 1 / np.median(np.diff(t))
        L, st = int(round(AP["win_s"] * fps)), max(1, int(round(AP["stride_s"] * fps)))
        X = np.nan_to_num(Xall[ii], nan=0.0)
        nanc = np.vstack([np.zeros(len(cols)), np.cumsum(~np.isfinite(Xall[ii]), 0)])
        C = np.vstack([np.zeros(len(cols)), np.cumsum(X, 0)])
        tc = t - t[0]
        Ct, Ctt = np.r_[0, np.cumsum(tc)], np.r_[0, np.cumsum(tc ** 2)]
        Cty = np.vstack([np.zeros(len(cols)), np.cumsum(X * tc[:, None], 0)])
        gapc = np.r_[0, np.cumsum(np.diff(t) > AP["gap_s"])]
        blk = g.block.to_numpy()
        b = bm[bm.subject_id == s].set_index("block")
        for k in np.unique(blk):
            pos = np.where(blk == k)[0]
            for a in range(pos[0], pos[-1] - L + 2, st):
                e = a + L
                if gapc[e - 1] - gapc[a] != 0:
                    continue
                n = L
                sy, st_, stt, sty = C[e] - C[a], Ct[e] - Ct[a], Ctt[e] - Ctt[a], Cty[e] - Cty[a]
                mean = sy / n
                slope = (sty - st_ * sy / n) / max(stt - st_ ** 2 / n, 1e-12)
                bad = (nanc[e] - nanc[a]) > 0
                mean[bad], slope[bad] = np.nan, np.nan
                bk = b.loc[k]
                tm = (t[a] + t[e - 1]) / 2
                third = min(2, int(3 * (tm - bk.t0) / max(bk.t1 - bk.t0, 1e-9)))
                recs.append((s, r, int(k), bk.phase, int(bk.pressure), t[a], t[e - 1], tm, third, int(ii[a]), L))
                mean_l.append(mean)
                slope_l.append(slope)
    W = pd.DataFrame(recs, columns=["subject_id", "roi_id", "block", "phase", "pressure", "t_start", "t_end", "t_mid",
                                    "third", "start_pos", "L"])
    W["y_phase"] = W.phase.map({"baseline": 0, "clamping": 1, "rest": 2}).astype(int)
    Xw = np.hstack([np.vstack(mean_l), np.vstack(slope_l)]).astype(np.float32)
    names = [f"{c}__mean" for c in cols] + [f"{c}__slope" for c in cols]
    return W, Xw, names, F


def set_columns(names, fset):
    fams = list(FAMILIES) if fset == "all" else [fset]
    keep = {c for fam in fams for c in FAMILIES[fam]}
    return [i for i, n in enumerate(names) if n.split("__")[0] in keep]


def segments(bm, s, k):
    b = bm[(bm.subject_id == s)]
    seg = []
    for _, bk in b.iterrows():
        L = bk.t1 - bk.t0
        seg.append((bk.t0 + k * L / 3, bk.t0 + (k + 1) * L / 3))
    return np.array(seg)


def purge_mask(Wsub, seg):
    lo = Wsub.t_start.to_numpy()[:, None]
    hi = Wsub.t_end.to_numpy()[:, None]
    a, b = seg[:, 0][None, :] - AP["purge_s"], seg[:, 1][None, :] + AP["purge_s"]
    return ~((hi >= a) & (lo <= b)).any(1)


def thirds_splits(W, idx, bm, ks=(0, 1, 2)):
    out = []
    Ws = W.loc[idx]
    for k in ks:
        te_parts, tr_mask = [], np.ones(len(Ws), bool)
        for s in Ws.subject_id.unique():
            ms = (Ws.subject_id == s).to_numpy()
            seg = segments(bm, s, k)
            te_parts.append(ms & (Ws.third == k).to_numpy())
            tr_mask &= ~ms | (purge_mask(Ws, seg) & (Ws.third != k).to_numpy())
        te = np.any(te_parts, 0)
        out.append((idx[tr_mask & ~te], idx[te]))
    return out


def fit_tuned(name, task, X, y, inner):
    grid = GRIDS.get(name, [{}])
    if len(grid) == 1 or not inner:
        best = grid[0]
    else:
        sc = []
        for hp in grid:
            v = []
            for tr, te in inner:
                if len(np.unique(y[tr])) < (2 if task == "cls" else 1) or len(te) == 0:
                    continue
                m = make_model(name, task, hp).fit(X[tr], y[tr])
                v.append(score(task, y[te], m.predict(X[te])))
            sc.append(np.mean(v) if v else -np.inf)
        best = grid[int(np.argmax(sc))]
    return make_model(name, task, best).fit(X, y), best


def done_keys(path):
    if not (RESUME and path.exists()):
        return {}
    out = {}
    for line in path.read_text().splitlines():
        try:
            r = json.loads(line)
            if r.get("config_hash") == CONFIG_HASH:
                out[r["key"]] = r
        except json.JSONDecodeError:
            continue
    return out


def append_jsonl(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(clean(rec), default=jdefault) + "\n")

# ===== SECTION: 28. S14 OCCLUSION-STATE MODELS, H2 AND GATED TEMPORAL MODELS =====
def run_classical(W, Xw, names, bm, res_path):
    done = done_keys(res_path)
    subs = sorted(W.subject_id.unique())
    jobs = []
    for task in ["cls", "reg"]:
        for fset in FEATURE_SETS:
            models = CLASSICAL_MODELS if fset in FULL_LADDER_SETS else sorted({"majority", H2_MODEL})
            for mname in models:
                for s in subs:
                    jobs.append(("V1", task, fset, mname, s, s))
                for a, b in itertools.permutations(subs, 2):
                    jobs.append(("V2", task, fset, mname, a, b))
    for task in ["cls", "reg"]:
        for a, b in itertools.permutations(subs, 2):
            jobs.append(("V2", task, "time", "time_only", a, b))
            jobs.append(("V2", task, "all", "shuffled_null", a, b))
    log.info(f"S14 classical: {len(jobs)} experiments ({len(done)} already done)")
    t0 = time.time()
    for n_job, (scheme, task, fset, mname, a, b) in enumerate(jobs):
        key = f"{scheme}|{task}|{fset}|{mname}|{a}|{b}"
        if key in done:
            continue
        cols = [0] if fset == "time" else set_columns(names, fset)
        X = W[["t_mid"]].to_numpy(np.float64) if fset == "time" else Xw[:, cols].astype(np.float64)
        y_all = W.y_phase.to_numpy() if task == "cls" else W.pressure.to_numpy(float)
        elig = np.ones(len(W), bool) if task == "cls" else (W.phase == "clamping").to_numpy()
        preds = []
        if scheme == "V1":
            base_idx = np.where(elig & (W.subject_id == a).to_numpy())[0]
            folds = thirds_splits(W, base_idx, bm)
            hps = []
            for k, (tr, te) in enumerate(folds):
                inner = [(np.searchsorted(tr, i_tr), np.searchsorted(tr, i_te))
                         for i_tr, i_te in thirds_splits(W, tr, bm, ks=[kk for kk in (0, 1, 2) if kk != k])]
                m, hp = fit_tuned(mname, task, X[tr], y_all[tr], inner)
                hps.append(hp)
                preds.append(pd.DataFrame(dict(widx=te, y=y_all[te], yhat=m.predict(X[te]), fold=k)))
            hp = hps
        else:
            tr = np.where(elig & (W.subject_id == a).to_numpy())[0]
            te = np.where(elig & (W.subject_id == b).to_numpy())[0]
            inner = [(np.searchsorted(tr, i_tr), np.searchsorted(tr, i_te)) for i_tr, i_te in thirds_splits(W, tr, bm)]
            ytr = y_all[tr].copy()
            mm = mname
            if mname == "shuffled_null":
                ytr = rng(key).permutation(ytr)
                mm = H2_MODEL
            m, hp = fit_tuned("rf" if mname == "time_only" else mm, task, X[tr], ytr, inner)
            preds.append(pd.DataFrame(dict(widx=te, y=y_all[te], yhat=m.predict(X[te]), fold=0)))
        P = pd.concat(preds, ignore_index=True)
        P.to_parquet(DIRS["predictions"] / f"{key.replace('|', '__')}.parquet", index=False)
        y, p = P.y.to_numpy(), P.yhat.to_numpy()
        ok = W.loc[P.widx]
        order = ok.roi_id.to_numpy() * 1e5 + ok.t_mid.to_numpy()
        met = metrics(task, y, p, ok.pressure.to_numpy() if task == "reg" else None)
        ci = mbb_metric_ci(task, y, p, order, tag=key)
        rec = dict(key=key, config_hash=CONFIG_HASH, scheme=scheme, task=task, feature_set=fset, model=mname,
                   train_subject=a, test_subject=b, n_test=len(P), hp=hp, ci_lo=ci[0], ci_hi=ci[1], **met)
        append_jsonl(res_path, rec)
        done[key] = rec
        if (n_job + 1) % 10 == 0:
            log.info(f"  classical {n_job + 1}/{len(jobs)} | {time.time() - t0:.0f} s | RSS {rss_mb():.0f} MB")
    return pd.DataFrame([done[f"{j[0]}|{j[1]}|{j[2]}|{j[3]}|{j[4]}|{j[5]}"] for j in jobs])


def run_spectro_lopo(res_path):
    SF = T("S8", "spectro_features")
    done = done_keys(res_path)
    y = SF.is160.to_numpy()
    groups = SF.subject_id.to_numpy()
    subs = sorted(np.unique(groups))
    sets = {"raw": RAW, "l1": L1, "ratio": RATIO, "all": RAW + L1 + RATIO}
    models = [m for m in CLASSICAL_MODELS if m != "pls"]
    for fset, cols in sets.items():
        X = SF[cols].to_numpy(np.float64)
        for mname in models:
            for s in subs:
                key = f"T1d|{fset}|{mname}|{s}"
                if key in done:
                    continue
                tr, te = groups != s, groups == s
                gk = GroupKFold(n_splits=3)
                inner = list(gk.split(X[tr], y[tr], groups[tr]))
                m, hp = fit_tuned(mname, "cls", X[tr], y[tr], inner)
                p = m.predict(X[te])
                rec = dict(key=key, config_hash=CONFIG_HASH, feature_set=fset, model=mname, test_subject=s, hp=hp,
                           bal_acc=balanced_accuracy_score(y[te], p), n_test=int(te.sum()))
                append_jsonl(res_path, rec)
                done[key] = rec
    return pd.DataFrame(list(done.values()))


def temporal_sequences(F, W, idx, cols, mu=None, sd=None):
    X = F[cols].to_numpy(np.float32)
    L = int(W.L.iloc[0])
    seq = np.stack([X[p:p + L] for p in W.start_pos.to_numpy()[idx]])
    if mu is None:
        mu, sd = np.nanmean(seq, (0, 1)), np.nanstd(seq, (0, 1)) + 1e-6
    seq = np.nan_to_num((seq - mu) / sd, nan=0.0).astype(np.float32)
    return seq, mu, sd


def train_temporal(backbone, task, Xtr, ytr, Xva, yva, seed, ckpt):
    set_determinism(seed)
    dev = torch.device(DEV)
    out = 3 if task == "cls" else 1
    model = BACKBONES[backbone](Xtr.shape[2], TRAIN["hidden"], out, TRAIN["dropout"]).to(dev)
    n_par = sum(p.numel() for p in model.parameters())
    if n_par > TRAIN["max_params"]:
        raise ValueError(f"{backbone} has {n_par} parameters > cap {TRAIN['max_params']}")
    opt = torch.optim.Adam(model.parameters(), lr=TRAIN["lr"], weight_decay=TRAIN["weight_decay"])
    if task == "cls":
        cw = np.bincount(ytr, minlength=3).astype(float)
        cw = torch.tensor(np.where(cw > 0, cw.sum() / (3 * np.maximum(cw, 1)), 0.0), dtype=torch.float32, device=dev)
        lossf = nn.CrossEntropyLoss(weight=cw)
        yt = torch.tensor(ytr, dtype=torch.long)
    else:
        lossf = nn.MSELoss()
        yt = torch.tensor(ytr / 160.0, dtype=torch.float32)[:, None]
    Xt = torch.tensor(Xtr)
    state = dict(epoch=0, best=-np.inf, best_state=None, bad=0, history=[])
    if RESUME and ckpt.exists():
        ck = torch.load(ckpt, map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        state = ck["state"]
        log.info(f"    resumed {ckpt.name} at epoch {state['epoch']}")
    g = torch.Generator().manual_seed(seed)
    while state["epoch"] < TRAIN["epochs"] and state["bad"] < TRAIN["patience"]:
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        tl = 0.0
        for i in range(0, len(Xt), TRAIN["batch_size"]):
            j = perm[i:i + TRAIN["batch_size"]]
            opt.zero_grad()
            loss = lossf(model(Xt[j].to(dev)), yt[j].to(dev))
            loss.backward()
            opt.step()
            tl += float(loss) * len(j)
        p = predict_temporal(model, Xva, task)
        sc = score(task, yva, p)
        state["epoch"] += 1
        state["history"].append(dict(epoch=state["epoch"], train_loss=tl / len(Xt), val_score=sc))
        if sc > state["best"]:
            state.update(best=sc, bad=0, best_state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        else:
            state["bad"] += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), state=state), ckpt)
    model.load_state_dict(state["best_state"])
    return model, state["history"], n_par


def predict_temporal(model, X, task, raw=False):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            outs.append(model(torch.tensor(X[i:i + 512]).to(DEV)).cpu().numpy())
    o = np.vstack(outs)
    if raw:
        return o
    return o.argmax(1) if task == "cls" else o[:, 0] * 160.0


def run_temporal(W, F, bm, clas, res_path):
    if not ENABLE_TEMPORAL or TORCH is None:
        log.warning("Temporal models not run (disabled or PyTorch unavailable)")
        return pd.DataFrame(), pd.DataFrame()
    done = done_keys(res_path)
    cols = FAMILIES[TEMPORAL_FEATURE_SET] if TEMPORAL_FEATURE_SET != "all" else FEATS
    subs = sorted(W.subject_id.unique())
    comps = []
    for task in ["cls", "reg"]:
        elig = np.ones(len(W), bool) if task == "cls" else (W.phase == "clamping").to_numpy()
        y_all = W.y_phase.to_numpy() if task == "cls" else W.pressure.to_numpy(float)
        for a, b in itertools.permutations(subs, 2):
            tr_all = np.where(elig & (W.subject_id == a).to_numpy())[0]
            te = np.where(elig & (W.subject_id == b).to_numpy())[0]
            tr, va = thirds_splits(W, tr_all, bm, ks=[2])[0]
            Xtr, mu, sd = temporal_sequences(F, W, tr, cols)
            Xva, _, _ = temporal_sequences(F, W, va, cols, mu, sd)
            Xte, _, _ = temporal_sequences(F, W, te, cols, mu, sd)
            for bb in TEMPORAL_BACKBONES:
                outs = []
                for seed in TRAIN["seeds"]:
                    key = f"temporal|{task}|{bb}|{a}|{b}|{seed}"
                    ck = DIRS["checkpoints"] / f"{key.replace('|', '__')}.pt"
                    if key in done:
                        o = np.load(DIRS["predictions"] / f"{key.replace('|', '__')}.npy")
                        outs.append(o)
                        continue
                    t0 = time.time()
                    m, hist, n_par = train_temporal(bb, task, Xtr, y_all[tr], Xva, y_all[va], SEED + seed, ck)
                    o = predict_temporal(m, Xte, task, raw=True)
                    np.save(DIRS["predictions"] / f"{key.replace('|', '__')}.npy", o)
                    torch.save(m.state_dict(), DIRS["models"] / f"{key.replace('|', '__')}.pt")
                    wjson(DIRS["models"] / f"{key.replace('|', '__')}_history.json", hist)
                    p = o.argmax(1) if task == "cls" else o[:, 0] * 160
                    met = metrics(task, y_all[te], p, W.pressure.to_numpy()[te] if task == "reg" else None)
                    rec = dict(key=key, config_hash=CONFIG_HASH, task=task, backbone=bb, train_subject=a, test_subject=b,
                               seed=seed, params=n_par, epochs=len(hist), seconds=time.time() - t0, **met)
                    append_jsonl(res_path, rec)
                    done[key] = rec
                    outs.append(o)
                    log.info(f"  temporal {key}: {'bal_acc' if task == 'cls' else 'rmse'}="
                             f"{met['bal_acc' if task == 'cls' else 'rmse']:.3f} ({time.time() - t0:.0f} s)")
                ens = np.mean(outs, 0)
                pe = ens.argmax(1) if task == "cls" else ens[:, 0] * 160
                ref = clas[(clas.scheme == "V2") & (clas.task == task) & (clas.train_subject == a) &
                           (clas.test_subject == b) & ~clas.model.isin(["time_only", "shuffled_null", "majority"])]
                bestc = ref.sort_values("bal_acc" if task == "cls" else "rmse", ascending=task != "cls").iloc[0]
                P = pd.read_parquet(DIRS["predictions"] / f"{bestc.key.replace('|', '__')}.parquet")
                pc = P.set_index("widx").loc[te, "yhat"].to_numpy()
                yy = y_all[te]
                order = W.roi_id.to_numpy()[te] * 1e5 + W.t_mid.to_numpy()[te]
                idx = np.argsort(order, kind="stable")
                r = rng(f"cmp{task}{bb}{a}{b}")
                diffs = []
                for _ in range(min(AP["boot"], 1000)):
                    j = idx[mbb_index(len(idx), AP["mbb_block"], r)]
                    diffs.append(score(task, yy[j], pe[j]) - score(task, yy[j], pc[j]))
                comps.append(dict(task=task, backbone=bb, train_subject=a, test_subject=b,
                                  ensemble_score=score(task, yy, pe), best_classical=bestc.key,
                                  classical_score=score(task, yy, pc), diff=score(task, yy, pe) - score(task, yy, pc),
                                  diff_ci_lo=np.percentile(diffs, 2.5), diff_ci_hi=np.percentile(diffs, 97.5)))
    comp = pd.DataFrame(comps)
    if len(comp):
        comp["better"] = comp.diff_ci_lo > 0
    return pd.DataFrame([v for v in done.values() if v["key"].startswith("temporal")]), comp


@step("S14", deps=["S8", "S9"], title="Occlusion-state models (T1a/T1b/T1d), H2, gated temporal models", critical=False)
def s14():
    FF, bm = T("S8", "frame_features"), T("S9", "block_meta")
    W, Xw, names, F = build_windows(FF, bm)
    log.info(f"Windows: {len(W):,} ({W.groupby('subject_id').size().to_dict()}); features {Xw.shape[1]}")
    W.to_parquet(DIRS["tables"] / "s14_windows.parquet", index=False)
    smoke = np.where((W.subject_id == W.subject_id.iloc[0]).to_numpy())[0][::10]
    if len(np.unique(W.y_phase.to_numpy()[smoke])) < 3 or not np.isfinite(Xw[smoke]).any():
        raise AssertionError("S14 smoke test: windows lack classes or finite features")
    sm_ = make_model(H2_MODEL, "cls", GRIDS[H2_MODEL][0]).fit(Xw[smoke][:, set_columns(names, "ratio")],
                                                              W.y_phase.to_numpy()[smoke])
    log.info(f"S14 smoke test passed ({len(smoke)} windows, in-sample bal_acc "
             f"{balanced_accuracy_score(W.y_phase.to_numpy()[smoke], sm_.predict(Xw[smoke][:, set_columns(names, 'ratio')])):.2f})")
    clas = run_classical(W, Xw, names, bm, DIRS["tables"] / "s14_classical_results.jsonl")
    spec = run_spectro_lopo(DIRS["tables"] / "s14_t1d_results.jsonl")
    v2 = clas[(clas.scheme == "V2") & (clas.task == "cls")]
    null = v2[v2.model == "shuffled_null"].set_index(["train_subject", "test_subject"]).bal_acc
    inf = []
    for (fs, m), g in v2[~v2.model.isin(["shuffled_null"])].groupby(["feature_set", "model"]):
        ok = all(r.ci_lo > null.loc[(r.train_subject, r.test_subject)] for r in g.itertuples())
        inf.append(dict(feature_set=fs, model=m, informative=ok, mean_bal_acc=g.bal_acc.mean()))
    inf = pd.DataFrame(inf)
    reg_v2 = clas[(clas.scheme == "V2") & (clas.task == "reg") & (clas.model == H2_MODEL) & (clas.feature_set == "all")]
    drift = []
    for r in reg_v2.itertuples():
        cols = set_columns(names, "all")
        tr = np.where(((W.subject_id == r.train_subject) & (W.phase == "clamping")).to_numpy())[0]
        te = np.where(((W.subject_id == r.test_subject) & (W.phase == "rest")).to_numpy())[0]
        hp = r.hp if isinstance(r.hp, dict) else GRIDS[H2_MODEL][0]
        m = make_model(H2_MODEL, "reg", hp).fit(Xw[tr][:, cols], W.pressure.to_numpy(float)[tr])
        p = m.predict(Xw[te][:, cols])
        rho, pv = stats.spearmanr(p, W.t_mid.to_numpy()[te])
        drift.append(dict(train_subject=r.train_subject, test_subject=r.test_subject, rho_pred_vs_time=rho, p=pv,
                          drift_flag=bool(rho > 0.3 and pv < AP["alpha"]),
                          mean_pred_rest=float(np.mean(p))))
    drift = pd.DataFrame(drift)
    h2c = v2[(v2.model == H2_MODEL) & v2.feature_set.isin(["raw", "ratio"])].pivot_table(
        index=["train_subject", "test_subject"], columns="feature_set", values="bal_acc")
    cam_ok = bool(((1 - h2c["ratio"]) < (1 - h2c["raw"])).all()) if {"raw", "ratio"} <= set(h2c.columns) else False
    sp = spec[spec.model == H2_MODEL].pivot_table(index="test_subject", columns="feature_set", values="bal_acc")
    dd = (1 - sp["ratio"]) - (1 - sp["raw"])
    p_t1d = wilcoxon_exact(dd.to_numpy(), "less")
    h2 = bool(cam_ok and p_t1d < AP["alpha"])
    tmp, comp = run_temporal(W, F, bm, clas, DIRS["tables"] / "s14_temporal_results.jsonl")
    # summary tables
    met_col = np.where(clas.task == "cls", clas.get("bal_acc", np.nan), clas.get("rmse", np.nan))
    bench = clas.assign(metric=met_col).groupby(["scheme", "task", "feature_set", "model"]).agg(
        mean=("metric", "mean"), lo=("ci_lo", "mean"), hi=("ci_hi", "mean"), n_runs=("metric", "size")).reset_index()
    bench["metric"] = np.where(bench.task == "cls", "balanced_accuracy", "rmse_mmHg")
    spec_sum = spec.groupby(["feature_set", "model"]).bal_acc.agg(["mean", "std", "min"]).reset_index()
    spec_sum["ci"] = [boot_ci(spec[(spec.feature_set == r.feature_set) & (spec.model == r.model)].bal_acc, tag=f"t1d{i}")
                      for i, r in enumerate(spec_sum.itertuples())]
    # figures
    sub = clas[(clas.scheme == "V2") & (clas.task == "cls") & ~clas.model.isin(["shuffled_null", "time_only"])]
    lab = sub.feature_set + " · " + sub.model
    order = sorted(lab.unique())
    fig, ax = plt.subplots(figsize=(7.4, max(3.0, 0.22 * len(order) + 1)))
    for j, (dirn, g) in enumerate(sub.groupby(["train_subject", "test_subject"])):
        gl = g.assign(l=g.feature_set + " · " + g.model).set_index("l").reindex(order)
        yy = np.arange(len(order)) + (j - 0.5) * 0.3
        ax.errorbar(gl.bal_acc, yy, xerr=[gl.bal_acc - gl.ci_lo, gl.ci_hi - gl.bal_acc], fmt="o", ms=3,
                    color=C_SUBJ[j], label=f"train {dirn[0][-2:]} → test {dirn[1][-2:]}", lw=0.8)
    for j, r in enumerate(clas[(clas.model == "shuffled_null") & (clas.task == "cls")].itertuples()):
        ax.axvline(r.bal_acc, color=C_SUBJ[j], ls=":", lw=0.8)
    ax.axvline(1 / 3, color="0.4", lw=0.6)
    ax.set(yticks=np.arange(len(order)), yticklabels=order, xlabel="Balanced accuracy (phase, cross-participant)")
    ax.legend(fontsize=7, loc="upper right")
    savefig(fig, "s14_benchmark_v2", "Cross-participant phase classification (T1a): balanced accuracy with "
            "moving-block bootstrap 95% CIs; dotted = shuffled-label null; solid grey = chance (1/3).")
    best = sub.sort_values("bal_acc", ascending=False).iloc[0]
    fig, ax = plt.subplots(figsize=(3.4, 3.0))
    cm = np.array(best.confusion)
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.6 else "black")
    ax.set(xticks=range(3), yticks=range(3), xticklabels=["base", "clamp", "rest"], yticklabels=["base", "clamp", "rest"],
           xlabel="Predicted", ylabel="True")
    ax.grid(False)
    savefig(fig, "s14_confusion_best", f"Row-normalised confusion matrix of the best cross-participant phase model "
            f"({best.feature_set} · {best.model}, train {best.train_subject[-2:]} → test {best.test_subject[-2:]}).")
    log.info(f"H2: camera ratio<raw in both directions={cam_ok}; T1d Wilcoxon p={p_t1d:.4g}; H2 supported={h2}")
    tabs = {"classical_results": clas, "benchmark": bench, "t1d_results": spec, "t1d_summary": spec_sum,
            "informative_gate": inf, "drift_check": drift}
    if len(tmp):
        tabs["temporal_results"] = tmp
    if len(comp):
        tabs["temporal_vs_classical"] = comp
    return R(tabs, info=dict(n_windows=len(W), h2=h2, h2_camera=cam_ok, h2_t1d_p=p_t1d,
                             h2_t1d_mean_diff=float(dd.mean()), drift_flags=int(drift.drift_flag.sum()) if len(drift) else 0,
                             temporal_better=int(comp.better.sum()) if len(comp) else 0,
                             temporal_run=bool(len(tmp)), best_v2=dict(feature_set=best.feature_set, model=best.model,
                                                                       bal_acc=best.bal_acc)))

# ===== SECTION: 29. REPORTING =====
def fmt(v, d=3):
    try:
        return f"{float(v):.{d}g}" if v is not None and np.isfinite(float(v)) else "n/a"
    except (TypeError, ValueError):
        return str(v)


def build_report():
    ok = lambda s: s in RESULTS and not RESULTS[s]["info"].get("_failed")
    summ = dict(code=CODE_VERSION, timestamp=TS, env=ENV, mode=RUN_MODE, profile=PROFILE, device=DEV,
                config_hash=CONFIG_HASH, data=DATA_INFO, plan_signed_off=PLAN_SIGNED_OFF, steps=STATUS,
                figures=FIGURES, outputs=str(OUT))
    L = [f"# PMHM Phase A–E report ({RUN_MODE} run, {TS})", "",
         f"Dataset: `{DATA_PATH.name}` ({DATA_INFO['rows']:,} records, SHA-256 `{DATA_INFO['sha256'][:16]}…`). "
         f"Environment: {ENV}, profile {PROFILE}, device {DEV}. Configuration hash `{CONFIG_HASH}`.", "",
         "**Scope.** SPECTRALPACA contains ten healthy volunteers imaged with 16 visible bands (465–641 nm). Camera "
         "features are ROI mean/SD statistics of tracked 20 × 20 px regions; they carry no pixel-level spatial "
         "information, so no spatial model is fitted. All camera-derived quantities are relative measures of "
         "haemoglobin-related absorption, not StO2 or perfusion, and nothing here validates PAD diagnosis.", "",
         f"**Analysis plan.** Frozen in S2 (hash `{I('S2')['plan_hash'] if ok('S2') else 'n/a'}`); supervisor sign-off: "
         f"{'yes' if PLAN_SIGNED_OFF else 'NO — confirmatory results are provisional'}.", "", "## Step status", "",
         md_table(pd.DataFrame([dict(step=k, title=v.get("title"), status=v["status"], seconds=v.get("seconds"),
                                     error=v.get("error", "")) for k, v in STATUS.items()]))]
    if ok("S1"):
        L += ["", "## S1 Contract and golden tests", "", f"{I('S1')['n_passed']}/{I('S1')['n_tests']} passed.", "",
              md_table(T("S1", "golden"))]
        summ["S1"] = I("S1")
    if ok("S3"):
        L += ["", "## S3 Quality control", "", md_table(T("S3", "qc_subject")), "", md_table(T("S3", "gaps"))]
        summ["S3"] = I("S3")
    if ok("S4"):
        i = I("S4")
        L += ["", "## S4 Dark offset", "", f"Sign flips of the primary-feature clamp deltas across dark scenarios: "
              f"{i['sign_flips']} (robust: {i['robust']}); maximum spread {fmt(i['max_spread'])}."]
        summ["S4"] = i
    if ok("S5"):
        i = I("S5")
        L += ["", "## S5 Forward model", "", f"Unit tests {i['tests']}; max truncation {fmt(i['max_truncation'])}; "
              f"negative spectrometer values {i['n_negative']} ({i['negative_in_support']} inside band support); "
              f"extinction design condition number {fmt(i['mbll_condition'])}."]
        summ["S5"] = i
    if ok("S6"):
        L += ["", "## S6 Relative reflectance and effective gain", "",
              f"Bands within ±{AP['gain_err_tol']:.0%} cross-participant prediction error: {I('S6')['bands_within_tol']}; "
              f"absolute calibration adopted: {I('S6')['adopt_absolute']}.", "", md_table(T("S6", "cross_prediction"))]
        summ["S6"] = I("S6")
    if ok("S7"):
        L += ["", "## S7 Cross-talk", "", f"Correction improved held-out spectral angle in {I('S7')['folds_improved']}/10 "
              f"folds; neutral-patch check passed: {I('S7')['neutral_pass']}; adopted as sensitivity variant: "
              f"{I('S7')['adopt_variant']}.", "", md_table(T("S7", "folds"))]
        summ["S7"] = I("S7")
    if ok("S10"):
        i = I("S10")
        L += ["", "## S10 Dose–response and H1", "", f"**H1: {i['h1']}.** Camera arm supported in "
              f"{i['camera_supported']}/2 participants (non-confirmatory). Spectrometer arm (n = 10): mean change of "
              f"log(S550/S609) {fmt(i['spectro_mean_diff'])}, exact one-sided Wilcoxon p = {fmt(i['spectro_p'])}, "
              f"95% CI {[fmt(x) for x in i['spectro_ci']]}.", "",
              md_table(T("S10", "h1_camera").drop(columns=[c for c in T("S10", "h1_camera") if c.startswith("d_")])), "",
              "Local change of the primary ratio by pressure (ROI mean):", "",
              md_table(T("S10", "h1_camera")[["subject_id"] + [c for c in T("S10", "h1_camera") if c.startswith("d_")]]),
              "", "Columns prefixed `exploratory_` were not pre-specified and must be reported as exploratory."]
        summ["S10"] = i
    if ok("S11"):
        i = I("S11")
        L += ["", "## S11 Recovery kinetics (RQ2)", "", f"Accepted exponential fits (primary feature, ROIs): "
              f"{i['accepted_share']:.0%} (model-based analysis: {i['model_based']}). Dependence on preceding pressure "
              f"(Holm p < {AP['alpha']} in both participants): {i['dependence']}.", "", md_table(T("S11", "rq2_tests"))]
        summ["S11"] = i
    if ok("S12"):
        L += ["", "## S12 Feature robustness (RQ3)", "", I("S12").get("note", ""), "", md_table(T("S12", "family_robustness")), "",
              md_table(T("S12", "rq3_verdict"))]
        summ["S12"] = I("S12")
    if ok("S13"):
        i = I("S13")
        L += ["", "## S13 Agreement and repeatability", "", f"Primary level: {i['primary_level']} "
              f"(ROI correspondence confirmed: {i['roi_correspondence_confirmed']}). {i['note']}", "",
              md_table(T("S13", "agreement_summary"))]
        summ["S13"] = i
    if ok("S14"):
        i = I("S14")
        L += ["", "## S14 Occlusion-state models and H2", "", f"Windows: {i['n_windows']:,}. **H2 supported: {i['h2']}** "
              f"(camera ratio<raw both directions: {i['h2_camera']}; T1d exact Wilcoxon p = {fmt(i['h2_t1d_p'])}). "
              f"Drift flags: {i['drift_flags']}. Temporal models run: {i['temporal_run']}; significantly better than "
              f"classical in {i['temporal_better']} comparisons.", "",
              md_table(T("S14", "benchmark").sort_values(["scheme", "task", "mean"], ascending=[True, True, False])), "",
              md_table(T("S14", "t1d_summary")), ""]
        if "temporal_vs_classical" in RESULTS["S14"]["tables"]:
            L += [md_table(T("S14", "temporal_vs_classical"))]
        summ["S14"] = i
    L += ["", "## Figures", ""] + [f"- `{Path(f['file']).name}`: {f['caption']}" for f in FIGURES]
    L += ["", "## Limitations", "",
          "- Camera spectra exist for two participants (seven ROIs); camera results are within-participant findings.",
          "- Pressure increases monotonically with time in the protocol; placebo and drift checks reduce but cannot "
          "remove this confound.",
          "- No white reference; relative reflectance and spectrometer-anchored gains replace absolute calibration.",
          "- Spectrometer spectra exist only at baseline and 160 mmHg and have no timestamps.",
          "- ROI correspondence between modalities is unconfirmed unless ROI_CORRESPONDENCE_CONFIRMED is set.",
          "- Healthy volunteers only; no clinical or diagnostic validity is implied.", "",
          "## Next steps", "",
          "- Supervisor sign-off of the frozen analysis plan; confirm ROI correspondence from the dataset publication.",
          "- S15 full temporal-model sweep and S16–S18 (short recordings, sensitivity battery, release).",
          "- Optional SX: rebuild the dataset file with all ten participants' camera spectra and re-run unchanged."]
    rep = DIRS["reports"] / f"PMHM_report_{RUN_MODE}_{TS}.md"
    rep.write_text("\n".join(L))
    wjson(DIRS["reports"] / f"experiment_summary_{RUN_MODE}_{TS}.json", summ)
    for k in ["S1", "S10", "S11", "S12", "S13", "S14"]:
        if ok(k):
            for name, df in RESULTS[k]["tables"].items():
                flat_table(df).to_csv(DIRS["tables"] / f"{k}_{name}.csv", index=False)
    log.info(f"Report: {rep}")
    return rep

# ===== SECTION: 30. MAIN EXECUTION =====
snapshot = dict(timestamp=TS, code=CODE_VERSION, env=ENV, profile=PROFILE, device=DEV, n_jobs=N_JOBS,
                ram_total_mb=TOTAL_MB, ram_avail_mb=AVAIL_MB, python=sys.version, data=DATA_INFO, paths=PATHS[ENV],
                data_path=str(DATA_PATH), config=CONFIG_SNAPSHOT, config_hash=CONFIG_HASH, quick=QUICK_TEST,
                resume=RESUME, experiments=EXPERIMENTS, plan_signed_off=PLAN_SIGNED_OFF)
wjson(DIRS["snapshots"] / f"config_snapshot_{TS}.json", snapshot)
targets = STEP_ORDER if EXPERIMENTS == "all" else list(EXPERIMENTS)
unknown = [t for t in targets if t not in STEPS]
if unknown:
    raise ValueError(f"Unknown steps in EXPERIMENTS: {unknown}")
event("main", "start", targets=targets)
T_MAIN = time.time()
try:
    for s_ in targets:
        run_step(s_)
finally:
    REPORT_PATH = build_report()
    event("main", "end", seconds=time.time() - T_MAIN)
    log.info(f"Finished in {time.time() - T_MAIN:.0f} s. Outputs in {OUT}")
    print(pd.DataFrame(STATUS).T.to_string())